In [ ]:
using Pkg; Pkg.gc() 

In [ ]:
Pkg.activate(".")

In [ ]:
using CUDA
CUDA.versioninfo()

# Config

In [ ]:
# Cellule 1: Changer de répertoire et charger le module
println("Re-loading NeuroDSL module after kernel fixes...")
# Important: on doit inclure depuis le dossier src/
include("src/NeuroDSL.jl")
using .NeuroDSL

println("--- Verification anti-duplication after reload ---")
checks = [
    (NeuroDSL.demand!,           "demand!",                    2),
    (NeuroDSL.backward_graph!,   "backward_graph!",            1),
    (NeuroDSL.execute_rule!,     "NeuroDSL.execute_rule!",     1),
    (NeuroDSL.accum_grad!,       "accum_grad!",                1),
]
all_ok = true
for (fn, name, max_m) in checks
    n = length(methods(fn))
    ok = n <= max_m
    global all_ok = all_ok && ok
    println(ok ? "✅" : "❌", " $name — $n méthode(s)")
end
if all_ok
    println("\n🚀 API chargée sans duplication.")
else
    error("❌ Duplication détectée !")
end

In [49]:
# ── TEST 1 — allowscalar(false) ne lève plus d'exception ──
import CUDA
CUDA.allowscalar(false)

logits_gpu = CUDA.cu(randn(Float32, 8, 16))
labels     = rand(1:16, 8)

loss = NeuroDSL.cross_entropy_loss(logits_gpu, labels)
grad = NeuroDSL.cross_entropy_grad(logits_gpu, labels)

@assert loss > 0f0            "loss doit être > 0"
@assert size(grad) == (8, 16) "shape du grad incorrecte"

println("✅ Test 1 — pas de scalar indexing GPU")
println("   loss = ", loss)
println("   size(grad) = ", size(grad))

✅ Test 1 — pas de scalar indexing GPU
   loss = 2.6472206
   size(grad) = (8, 16)


In [ ]:
using CUDA

# List all GPUs – CUDA.devices() returns a vector of CuDevice objects
for (i, dev) in enumerate(CUDA.devices())
    println("$i: $(CUDA.name(dev))")
end

In [ ]:
CUDA.device!(1)  

In [ ]:
println("Active GPU: ", CUDA.name(CUDA.device()))

In [2]:
function rope_grad_check(; M=4, N=8, eps=1f-3, tol=5f-2)
    g = NeuroDSL.JuliusGraph(namespace=:t_rope)
    NeuroDSL.set!(g, :x,    NeuroDSL.Backend.randn32(g.device, M, N); is_param=true)
    NeuroDSL.set!(g, :Z,    NeuroDSL.Backend.zeros32(g.device, M, N); atom_type=NeuroDSL.Datom)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:r,  [:x], :rope;     namespace=:t_rope))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L,  [:r, :Z], :mse_loss; namespace=:t_rope))

    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, :L; ctx_store=ctx)
    NeuroDSL.backward_graph!(g, :L; ctx_store=ctx)
    grad_a = Array(NeuroDSL.node(g, :x).gradient)

    pn = NeuroDSL.node(g, :x)
    orig = copy(pn.value); orig_cpu = Array(orig)
    grad_n = zeros(Float32, size(orig))
    for i in eachindex(orig_cpu)
        v⁺ = copy(orig_cpu); v⁺[i] += eps
        pn.value = NeuroDSL.Backend.to_device(g.device, v⁺)
        NeuroDSL.invalidate_all!(g)
        l⁺ = sum(Array(NeuroDSL.demand!(g, :L)))
        v⁻ = copy(orig_cpu); v⁻[i] -= eps
        pn.value = NeuroDSL.Backend.to_device(g.device, v⁻)
        NeuroDSL.invalidate_all!(g)
        l⁻ = sum(Array(NeuroDSL.demand!(g, :L)))
        grad_n[i] = (l⁺ - l⁻) / (2eps)
    end
    pn.value = orig; NeuroDSL.invalidate_all!(g)

    diff = maximum(abs.(grad_a .- grad_n))
    ok   = diff < tol
    println(ok ? "✅" : "❌", " RoPE grad check — max diff = ", round(diff, digits=6))
    return ok
end

rope_grad_check()

✅ RoPE grad check — max diff = 4.7e-5


true

# Zygote

In [3]:
using Zygote, Printf

# Version de RMSNorm compatible Zygote (évite les broadcasts ambigus)
function rmsnorm_zygote(x::Matrix{Float32}, gamma::Vector{Float32}, eps::Float32=1f-6)
    nc = size(x, 2)
    ms = sum(x.^2, dims=2) ./ Float32(nc) .+ eps
    rms_inv = @. 1f0 / sqrt(ms)          # force un broadcast élémentaire
    return x .* rms_inv .* gamma'        # (M,N) .* (M,1) .* (1,N)
end

function llama_step_zygote(x, w, gamma)
    h = rmsnorm_zygote(x, gamma)
    s = h * w'
    return sum(s)
end

let
    M, N = 8, 16
    dev = NeuroDSL.Backend.CPUDevice()   # forcer CPU pour Zygote

    x_val = NeuroDSL.Backend.randn32(dev, M, N)
    w_val = NeuroDSL.Backend.randn32(dev, N, N)
    g_val = NeuroDSL.Backend.ones32(dev, N)

    # S'assurer que ce sont bien des Array Julia (pas CuArray)
    x_arr = Array(x_val)
    w_arr = Array(w_val)
    g_arr = Array(g_val)

    # ---- Côté NeuroDSL ----
    g = NeuroDSL.JuliusGraph(namespace=:zygote_bench, device=dev)
    NeuroDSL.set!(g, :X, x_arr)
    NeuroDSL.set!(g, :W, w_arr; is_param=true)
    NeuroDSL.set!(g, :gamma, g_arr; is_param=true)

    NeuroDSL.@addrules g :zygote_bench begin
        H       = rmsnorm(X, gamma)
        Out_mat = matmul(H, W, trans_b=true)
        Loss    = sum_matrix(Out_mat)
    end
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, :Loss; ctx_store=ctx)
    NeuroDSL.backward_graph!(g, :Loss; ctx_store=ctx)

    dW_dsl = NeuroDSL.node(g, :W).gradient
    dG_dsl = NeuroDSL.node(g, :gamma).gradient

    # ---- Côté Zygote ----
    loss_zyg, grads = Zygote.withgradient(llama_step_zygote, x_arr, w_arr, g_arr)
    dW_zyg = grads[2]
    dG_zyg = grads[3]

    # ---- Comparaison ----
    diff_W = maximum(abs.(dW_dsl .- dW_zyg))
    diff_G = maximum(abs.(dG_dsl .- dG_zyg))

    @printf "  W Max Diff=%.6f | G Max Diff=%.6f\n" diff_W diff_G
    if diff_W < 1e-4 && diff_G < 1e-4
        println("✅ Les gradients NeuroDSL et Zygote sont identiques !")
    else
        println("❌ Écart détecté. Vérifiez l'implémentation.")
    end
end

✅ @addrules [:zygote_bench] — 3 règles
  W Max Diff=0.000000 | G Max Diff=0.000000
✅ Les gradients NeuroDSL et Zygote sont identiques !


In [19]:
using Statistics, Printf

# Recrée le graphe
dev = NeuroDSL.Backend.CPUDevice()
g = NeuroDSL.JuliusGraph(device=dev)
NeuroDSL.set!(g, :x, randn(Float32, 1024, 1024))
out = NeuroDSL.LlamaBlock(1024, 16, 4096)(g, :x, :b)
NeuroDSL.set!(g, :y, zeros(Float32, 1024, 1024); atom_type=NeuroDSL.Datom)
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [out, :y], :mse_loss))

ctx = NeuroDSL.CtxStore()
for _ in 1:5  # warmup
    NeuroDSL.invalidate_all!(g)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
end

using Statistics, Printf
gc_times = Float64[]; bwd_times = Float64[]
for _ in 1:10
    NeuroDSL.invalidate_all!(g)
    NeuroDSL.demand!(g, :loss; ctx_store=ctx)
    GC.gc()
    s = @timed NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
    push!(gc_times,  s.gctime * 1000)
    push!(bwd_times, s.time   * 1000)
end

@printf "GC time  median : %.2f ms\n" median(gc_times)
@printf "backward median : %.2f ms\n" median(bwd_times)
@printf "backward min    : %.2f ms\n" minimum(bwd_times)

GC time  median : 23.71 ms
backward median : 2823.89 ms
backward min    : 2621.46 ms


In [ ]:
import NeuroDSL: mse_loss_bwd

# Redéfinition GPU‑compatible (extraction scalaire sécurisée)
function mse_loss_bwd(out, target, dy)
    # dy est un scalaire (loss) mais potentiellement sur GPU
    if dy isa AbstractArray && length(dy) == 1
        dy_scalar = CUDA.@allowscalar Array(dy)[1]
    else
        dy_scalar = dy
    end
    return (2f0 / length(out)) .* (out .- target) .* dy_scalar
end

In [23]:



g_gpu = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice())
NeuroDSL.set!(g_gpu, :x, CUDA.randn(Float32, 1024, 1024))
out = NeuroDSL.LlamaBlock(1024, 16, 4096)(g_gpu, :x, :b)
NeuroDSL.set!(g_gpu, :y, CUDA.zeros(Float32, 1024, 1024); atom_type=NeuroDSL.Datom)
NeuroDSL.addrule!(g_gpu, NeuroDSL.GraphRule(:loss, [out, :y], :mse_loss))
CUDA.allowscalar(true)
for _ in 1:5
    NeuroDSL.invalidate_all!(g_gpu)
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g_gpu, :loss; ctx_store=ctx)
    NeuroDSL.backward_graph!(g_gpu, :loss; ctx_store=ctx)
    CUDA.synchronize()
end

using Statistics, Printf
times_gpu = Float64[]
for _ in 1:10
    NeuroDSL.invalidate_all!(g_gpu)
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g_gpu, :loss; ctx_store=ctx)
    t = @elapsed begin
        NeuroDSL.backward_graph!(g_gpu, :loss; ctx_store=ctx)
        CUDA.synchronize()
    end
    push!(times_gpu, t * 1000)
end

@printf "GPU backward median : %.2f ms\n" median(times_gpu)
@printf "GPU backward min    : %.2f ms\n" minimum(times_gpu)

┌ Warning: It's not recommended to use allowscalar([true]) to allow scalar indexing.
│ Instead, use `allowscalar() do end` or `@allowscalar` to denote exactly which operations can use scalar operations.
└ @ GPUArraysCore C:\Users\Nevermind\.julia\packages\GPUArraysCore\aNaXo\src\GPUArraysCore.jl:184


GPU backward median : 38.17 ms
GPU backward min    : 27.13 ms


In [52]:
# ── TEST CPU vs GPU — Cohérence numérique ──
println("\n=== TEST CPU vs GPU ===")

# Mêmes données
logits_cpu = randn(Float32, 8, 16)
logits_gpu = CUDA.cu(logits_cpu)
labels = rand(1:16, 8)

# CPU
loss_cpu = NeuroDSL.cross_entropy_loss(logits_cpu, labels)
grad_cpu = NeuroDSL.cross_entropy_grad(logits_cpu, labels)

# GPU
loss_gpu = NeuroDSL.cross_entropy_loss(logits_gpu, labels)
grad_gpu = Array(NeuroDSL.cross_entropy_grad(logits_gpu, labels))

# Comparaison
diff_loss = abs(loss_cpu - loss_gpu)
diff_grad = maximum(abs.(grad_cpu .- grad_gpu))

println("Différence loss CPU/GPU : ", diff_loss)
println("Différence grad CPU/GPU : ", diff_grad)

if diff_loss < 1e-5 && diff_grad < 1e-5
    println("✅ CPU et GPU donnent des résultats IDENTIQUES")
else
    println("⚠️  Légère différence numérique normale en Float32")
end


=== TEST CPU vs GPU ===
Différence loss CPU/GPU : 0.0
Différence grad CPU/GPU : 3.7252903e-9
✅ CPU et GPU donnent des résultats IDENTIQUES


In [53]:
# ── TEST END-TO-END — loss descend sur 50 steps ──
let
    dev = NeuroDSL.Backend.CPUDevice()
    g   = NeuroDSL.JuliusGraph(namespace=:e2e, device=dev)

    # Données fixes
    B, D = 8, 16
    NeuroDSL.set!(g, :X, NeuroDSL.Backend.randn32(dev, B, D))
    NeuroDSL.set!(g, :Y, NeuroDSL.Backend.randn32(dev, B, D); atom_type=NeuroDSL.Datom)

    # Modèle : Linear → RMSNorm → Linear → MSE
    h1   = NeuroDSL.Linear(D, D)(g, :X,  :fc1;  namespace=:e2e)
    h2   = NeuroDSL.LayerNorm(D)(g, h1,  :norm; namespace=:e2e)
    h3   = NeuroDSL.Linear(D, D)(g, h2,  :fc2;  namespace=:e2e)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [h3, :Y], :mse_loss; namespace=:e2e))

    # État AdamW par paramètre
    ps  = NeuroDSL.params(g; namespace=:e2e)
    m1s = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    m2s = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    lr, b1, b2, epsv, wd, clip = 1f-3, 0.9f0, 0.999f0, 1f-8, 1f-2, 1f0

    losses = Float32[]

    for t in 1:50
        ctx = NeuroDSL.CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        push!(losses, Array(NeuroDSL.node(g, :loss).value)[1])

        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)

        for (i, p) in enumerate(ps)
            NeuroDSL.adamw_step!(dev, p.value, p.gradient,
                                  m1s[i], m2s[i],
                                  lr, b1, b2, epsv, t, clip, wd)
        end

        NeuroDSL.invalidate_all!(g; namespace=:e2e)
    end

    # ── Assertions ──
    l1, l50 = losses[1], losses[end]
    reduction = (l1 - l50) / l1

    @assert l50 < l1        "loss n'a pas diminué : $l1 → $l50"
    @assert reduction > 0.3 "réduction trop faible : $(round(reduction*100, digits=1))%"

    # ── Résumé ──
    println("✅ Test end-to-end — loss descend sur 50 steps")
    println("   loss[1]   = ", round(l1,  digits=5))
    println("   loss[50]  = ", round(l50, digits=5))
    println("   réduction = ", round(reduction * 100, digits=1), "%")
    println()

    # ── Courbe ASCII ──
    println("   Courbe de loss (every 5 steps) :")
    for i in 1:5:50
        bar = "█" ^ Int(round(losses[i] / l1 * 20))
        @printf "   step %2d | %s %.4f\n" i bar losses[i]
    end
end

✅ Test end-to-end — loss descend sur 50 steps
   loss[1]   = 1.08068
   loss[50]  = 0.47629
   réduction = 55.9%

   Courbe de loss (every 5 steps) :
   step  1 | ████████████████████ 1.0807
   step  6 | ██████████████████ 0.9835
   step 11 | █████████████████ 0.8971
   step 16 | ███████████████ 0.8208
   step 21 | ██████████████ 0.7535
   step 26 | █████████████ 0.6939
   step 31 | ████████████ 0.6405
   step 36 | ███████████ 0.5922
   step 41 | ██████████ 0.5480
   step 46 | █████████ 0.5071


In [54]:
# ── TEST Multi-thread / Réentrance ──
using Printf
println("="^50)
println("   Test multi-thread / réentrance")
println("="^50)
println("Threads disponibles : ", Threads.nthreads())

# ── Test 1 : Réentrance séquentielle ──
# Plusieurs appels demand! sur le même graphe → résultats stables
println("\n--- Test 1 : Réentrance séquentielle ---")
let
    dev = NeuroDSL.Backend.CPUDevice()
    g   = NeuroDSL.JuliusGraph(namespace=:reentrant, device=dev)
    NeuroDSL.set!(g, :x,     ones(Float32, 4, 4))
    NeuroDSL.set!(g, :gamma, ones(Float32, 4); is_param=true)
    NeuroDSL.@addrules g :reentrant begin
        h = rmsnorm(x, gamma)
    end

    results = [Array(NeuroDSL.demand!(g, :h; namespace=:reentrant)) for _ in 1:10]
    @assert all(results[i] ≈ results[1] for i in 2:10) "résultats inconsistants"
    println("✅ 10 appels successifs à demand! — résultats identiques")
end

# ── Test 2 corrigé : Isolation entre graphes ──
println("\n--- Test 2 : Isolation entre graphes ---")
let
    dev = NeuroDSL.Backend.CPUDevice()

    # Données différentes (pas scale-invariant avec add)
    x1 = randn(Float32, 4, 4)
    x2 = randn(Float32, 4, 4) .+ 10f0   # décalage clair

    g1 = NeuroDSL.JuliusGraph(namespace=:iso1, device=dev)
    NeuroDSL.set!(g1, :x, x1)
    NeuroDSL.set!(g1, :w, ones(Float32, 4, 4))
    NeuroDSL.@addrules g1 :iso1 begin; h = add(x, w); end

    g2 = NeuroDSL.JuliusGraph(namespace=:iso2, device=dev)
    NeuroDSL.set!(g2, :x, x2)
    NeuroDSL.set!(g2, :w, ones(Float32, 4, 4))
    NeuroDSL.@addrules g2 :iso2 begin; h = add(x, w); end

    r1 = Array(NeuroDSL.demand!(g1, :h; namespace=:iso1))
    r2 = Array(NeuroDSL.demand!(g2, :h; namespace=:iso2))

    @assert !all(r1 .≈ r2)    "les graphes partagent des buffers !"
    @assert all(isfinite, r1) "g1 contient des NaN/Inf"
    @assert all(isfinite, r2) "g2 contient des NaN/Inf"

    # Vérification que chaque graphe a bien calculé sa propre valeur
    @assert r1 ≈ x1 .+ 1f0   "g1 : résultat incorrect"
    @assert r2 ≈ x2 .+ 1f0   "g2 : résultat incorrect"

    println("✅ Deux graphes distincts — pas d'interférence")
    println("   sum(r1) = ", round(sum(r1), digits=4),
            " | sum(r2) = ", round(sum(r2), digits=4))
end

# ── Test 3 : Concurrence réelle ──
println("\n--- Test 3 : Multi-thread concurrent ---")
if Threads.nthreads() == 1
    println("⚠️  1 thread seulement — test concurrent ignoré")
    println("   Pour activer : ajouter en cellule shell avant le kernel Julia")
    println("   ENV[\"JULIA_NUM_THREADS\"] = \"4\"")
else
    n = Threads.nthreads() * 2

    # Référence séquentielle
    ref = map(1:n) do i
        dev = NeuroDSL.Backend.CPUDevice()
        g   = NeuroDSL.JuliusGraph(namespace=Symbol(:seq, i), device=dev)
        NeuroDSL.set!(g, :x,     ones(Float32, 4, 4) .* Float32(i))
        NeuroDSL.set!(g, :gamma, ones(Float32, 4); is_param=true)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h, [:x, :gamma], :rmsnorm;
                           namespace=Symbol(:seq, i)))
        sum(Array(NeuroDSL.demand!(g, :h; namespace=Symbol(:seq, i))))
    end

    # Évaluation parallèle
    tasks = map(1:n) do i
        Threads.@spawn begin
            dev = NeuroDSL.Backend.CPUDevice()
            g   = NeuroDSL.JuliusGraph(namespace=Symbol(:par, i), device=dev)
            NeuroDSL.set!(g, :x,     ones(Float32, 4, 4) .* Float32(i))
            NeuroDSL.set!(g, :gamma, ones(Float32, 4); is_param=true)
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:h, [:x, :gamma], :rmsnorm;
                               namespace=Symbol(:par, i)))
            sum(Array(NeuroDSL.demand!(g, :h; namespace=Symbol(:par, i))))
        end
    end
    par = fetch.(tasks)

    diffs = abs.(ref .- par)
    @assert all(diffs .< 1f-4) "divergence en multi-thread : max=$(maximum(diffs))"
    println("✅ $(Threads.nthreads()) threads — résultats identiques au séquentiel")
    println("   max diff seq/par = ", maximum(diffs))
end

# ── Résumé des risques résiduels ──
println("\n--- Risques résiduels non testables ici ---")
println("⚠️  _MASK_CACHE : Dict global sans lock — écriture concurrente non sûre")
println("   → Safe si un seul thread écrit (warm-up séquentiel avant @threads)")
println("⚠️  _BUFFER_POOL : inutilisé pour les nœuds (fix 3) mais reste non thread-safe")
println("   → Peut être supprimé sans impact si multi-thread requis")

   Test multi-thread / réentrance
Threads disponibles : 4

--- Test 1 : Réentrance séquentielle ---
✅ @addrules [:reentrant] — 1 règles
✅ 10 appels successifs à demand! — résultats identiques

--- Test 2 : Isolation entre graphes ---
✅ @addrules [:iso1] — 1 règles
✅ @addrules [:iso2] — 1 règles
✅ Deux graphes distincts — pas d'interférence
   sum(r1) = 19.414 | sum(r2) = 178.0946

--- Test 3 : Multi-thread concurrent ---
✅ 4 threads — résultats identiques au séquentiel
   max diff seq/par = 0.0

--- Risques résiduels non testables ici ---
⚠️  _MASK_CACHE : Dict global sans lock — écriture concurrente non sûre
   → Safe si un seul thread écrit (warm-up séquentiel avant @threads)
⚠️  _BUFFER_POOL : inutilisé pour les nœuds (fix 3) mais reste non thread-safe
   → Peut être supprimé sans impact si multi-thread requis


In [ ]:
Threads.nthreads()  # Doit afficher 4

In [55]:
# ── BENCHMARK MÉMOIRE PEAK À SCALE ──
using Printf
#NeuroDSL.Backend.CUDA_AVAILABLE && import CUDA

function build_model(dev, M, D; ns)
    g = NeuroDSL.JuliusGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :X, NeuroDSL.Backend.randn32(dev, M, D))
    NeuroDSL.set!(g, :Y, NeuroDSL.Backend.zeros32(dev, M, D); atom_type=NeuroDSL.Datom)
    h1 = NeuroDSL.Linear(D, D)(g, :X,  :fc1;  namespace=ns)
    h2 = NeuroDSL.LayerNorm(D)(g, h1,  :norm; namespace=ns)
    h3 = NeuroDSL.Linear(D, D)(g, h2,  :fc2;  namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [h3, :Y], :mse_loss; namespace=ns))
    return g
end

function measure_allocs(g, ns)
    # warmup
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=ns)
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    GC.gc()

    # forward
    fwd = @allocated begin
        ctx = NeuroDSL.CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=ns)
    end

    # backward
    bwd = @allocated begin
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=ns)
    end

    NeuroDSL.invalidate_all!(g; namespace=ns)
    return fwd, bwd
end

# ── CPU ──
println("="^60)
println("   CPU — allocations forward / backward par taille")
println("="^60)
@printf "%-20s %12s %12s %12s\n" "config (M×D)" "fwd (KB)" "bwd (KB)" "total (KB)"
println("-"^60)

for (M, D) in [(8,64), (32,128), (128,256), (512,512), (1024,512)]
    ns  = Symbol(:bench_cpu, M, :_, D)
    dev = NeuroDSL.Backend.CPUDevice()
    g   = build_model(dev, M, D; ns=ns)
    fwd, bwd = measure_allocs(g, ns)
    @printf "%-20s %12.1f %12.1f %12.1f\n" "$(M)×$(D)" fwd/1024 bwd/1024 (fwd+bwd)/1024
end

# ── Fuite mémoire CPU ──
println("\n--- Test fuite mémoire CPU (10 runs, M=128 D=256) ---")
let
    ns  = :leak_test
    dev = NeuroDSL.Backend.CPUDevice()
    g   = build_model(dev, 128, 256; ns=ns)
    allocs = map(1:10) do _
        a = @allocated begin
            ctx = NeuroDSL.CtxStore()
            NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=ns)
            NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=ns)
            NeuroDSL.invalidate_all!(g; namespace=ns)
        end
        a
    end
    drift = maximum(allocs) - minimum(allocs)
    @printf "   allocs min=%.1f KB  max=%.1f KB  drift=%.1f KB\n" minimum(allocs)/1024 maximum(allocs)/1024 drift/1024
    drift < 500*1024 ? println("✅ Pas de fuite mémoire détectée") :
                       println("⚠️  Drift élevé : possible fuite")
end

# ── CUDA ──
if NeuroDSL.Backend.CUDA_AVAILABLE
    println("\n", "="^60)
    println("   CUDA — mémoire GPU utilisée par taille")
    println("="^60)
    @printf "%-20s %12s %12s %12s\n" "config (M×D)" "avant (MB)" "après (MB)" "delta (MB)"
    println("-"^60)

    for (M, D) in [(8,64), (32,128), (128,256), (512,512)]
        ns  = Symbol(:bench_gpu, M, :_, D)
        dev = NeuroDSL.Backend.CUDADevice()
        GC.gc(); CUDA.reclaim()
        mem_before = CUDA.used_memory()
        g   = build_model(dev, M, D; ns=ns)
        ctx = NeuroDSL.CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx, namespace=ns)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, namespace=ns)
        mem_after = CUDA.used_memory()
        @printf "%-20s %12.2f %12.2f %12.2f\n" "$(M)×$(D)" mem_before/1024^2 mem_after/1024^2 (mem_after-mem_before)/1024^2
    end
end

   CPU — allocations forward / backward par taille
config (M×D)             fwd (KB)     bwd (KB)   total (KB)
------------------------------------------------------------
8×64                          5.8         91.5         97.2
32×128                       19.8        424.7        444.4
128×256                     131.7       2315.3       2447.0
512×512                    1027.8      14355.2      15383.0
1024×512                   2051.8      24597.2      26649.0

--- Test fuite mémoire CPU (10 runs, M=128 D=256) ---
   allocs min=2447.0 KB  max=2835.0 KB  drift=388.0 KB
✅ Pas de fuite mémoire détectée

   CUDA — mémoire GPU utilisée par taille
config (M×D)           avant (MB)   après (MB)   delta (MB)
------------------------------------------------------------
8×64                       754.10       754.26         0.16
32×128                     754.14       754.90         0.76
128×256                    754.59       758.60         4.01
512×512                    757.03       78

In [56]:
function profile_backward_allocs_detailed(g::NeuroDSL.JuliusGraph, loss_sym::Symbol; namespace=g.active_ns)
    # Forward
    ctx_store = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx_store, namespace=namespace)

    # Réinitialisation des gradients
    NeuroDSL.zero_grads!(g; namespace=namespace)
    ln = NeuroDSL.node(g, loss_sym; namespace=namespace)
    ln.gradient = NeuroDSL.Backend.ones32(g.device, size(ln.value)...)

    println("\n--- Detailed Backward Allocations (KB) ---")
    println(lpad("Operation", 20), lpad("Allocations (KB)", 20))
    println("-"^(40))

    total_allocs = 0
    for out_sym in reverse(NeuroDSL.topo_order!(g; namespace=namespace))
        !haskey(g.rules[namespace], out_sym) && continue
        rule = g.rules[namespace][out_sym]
        nd_out = g.nodes[namespace][out_sym]
        nd_out.gradient === nothing && continue

        if !haskey(NeuroDSL.GRAD_RULES, rule.op)
            @warn "No backward rule for :$(rule.op), skipping profiling."
            continue
        end

        ctx = get(ctx_store, out_sym, Dict{Symbol,Any}())
        inputs_vals = [g.nodes[namespace][s].value for s in rule.inputs]

        # Mesurer les allocations pour cette règle + accumulation
        allocs = @allocated begin
            grads = NeuroDSL.GRAD_RULES[rule.op](g.device, nd_out.gradient, ctx, inputs_vals)
            for (i, in_sym) in enumerate(rule.inputs)
                NeuroDSL.accum_grad!(g.nodes[namespace][in_sym], grads[i])
            end
        end

        total_allocs += allocs
        @printf "%s %19.1f\n" lpad("$(rule.op)", 20) allocs/1024

        # Optionnel : libérer le gradient pour économiser la mémoire
        nd_out.gradient = nothing
    end
    println("-"^(40))
    @printf "Total allocations: %.1f KB\n" total_allocs/1024

    NeuroDSL.invalidate_all!(g; namespace=namespace)
    GC.gc()
end
# Re-using the setup from the memory benchmark (Cell 2VSj1PAxy2WS)
let
    dev = NeuroDSL.Backend.CPUDevice()
    ns  = :bench_fix_detailed
    g   = NeuroDSL.JuliusGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :X, NeuroDSL.Backend.randn32(dev, 128, 256))
    NeuroDSL.set!(g, :Y, NeuroDSL.Backend.zeros32(dev, 128, 256); atom_type=NeuroDSL.Datom)
    h1 = NeuroDSL.Linear(256, 256)(g, :X,  :fc1;  namespace=ns)
    h2 = NeuroDSL.LayerNorm(256)(g, h1,    :norm; namespace=ns)
    h3 = NeuroDSL.Linear(256, 256)(g, h2,  :fc2;  namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [h3, :Y], :mse_loss; namespace=ns))

    profile_backward_allocs_detailed(g, :loss; namespace=ns)
end


--- Detailed Backward Allocations (KB) ---
           Operation    Allocations (KB)
----------------------------------------
            mse_loss               384.2
              linear               770.6
             rmsnorm               386.9
              linear               770.6
----------------------------------------
Total allocations: 2312.2 KB


# Test

In [57]:
# Test 1
using Test
include("test/test_graph_api.jl")

Test Summary: | Pass  Total  Time
Graph API     |    7      7  1.5s


Test.DefaultTestSet("Graph API", Any[Test.DefaultTestSet("set! / node", Any[], 3, false, false, true, 1.781249881615e9, 1.781249883002e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_graph_api.jl"), Test.DefaultTestSet("addrule! / topo_order!", Any[], 2, false, false, true, 1.781249883002e9, 1.78124988306e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_graph_api.jl"), Test.DefaultTestSet("invalidation", Any[], 1, false, false, true, 1.78124988306e9, 1.781249883069e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_graph_api.jl"), Test.DefaultTestSet("namespace isolation", Any[], 1, false, false, true, 1.781249883069e9, 1.781249883097e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_graph_api.jl")], 0, false, false, true, 1.781249881571e9, 1.781249883097e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_graph_api.jl")

In [58]:
# Test 2
include("test/test_kernels.jl")

Test Summary: | Pass  Total  Time
Kernels       |   15     15  0.7s


Test.DefaultTestSet("Kernels", Any[Test.DefaultTestSet("RMSNorm fwd/bwd CPU", Any[], 4, false, false, true, 1.781249886482e9, 1.781249886721e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_kernels.jl"), Test.DefaultTestSet("SwiGLU fwd/bwd CPU", Any[], 2, false, false, true, 1.781249886721e9, 1.781249886721e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_kernels.jl"), Test.DefaultTestSet("Softmax", Any[], 2, false, false, true, 1.781249886721e9, 1.781249887103e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_kernels.jl"), Test.DefaultTestSet("MSE loss", Any[], 5, false, false, true, 1.781249887103e9, 1.781249887141e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_kernels.jl"), Test.DefaultTestSet("Cross-entropy", Any[], 2, false, false, true, 1.781249887141e9, 1.781249887142e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_kernels.jl")], 0, false, false, true, 1.781249886482e9, 1.781249887142e9, false, "c:\\Use

In [59]:
# Test 3
using Test
include("test/test_backward.jl")

✅ @addrules [:t_rmsnorm] — 2 règles
  [:gamma] max_err=4.56e-04  mean_err=1.81e-04  (tol=1e-03) ✅
  [:A] max_err=7.35e-05  mean_err=3.09e-05  (tol=5e-03) ✅
  [:A] max_err=5.39e-05  mean_err=3.73e-05  (tol=5e-03) ✅
  [:fc_W] max_err=7.46e-05  mean_err=2.66e-05  (tol=1e-03) ✅
  [:fc_b] max_err=4.21e-05  mean_err=1.68e-05  (tol=1e-03) ✅
✅ @addrules [:t_sg] — 2 règles
  [:gate] max_err=7.50e-06  mean_err=2.11e-06  (tol=5e-03) ✅
✅ @addrules [:t_sm] — 2 règles
  [:x] max_err=7.11e-05  mean_err=3.68e-05  (tol=1e-03) ✅
  [:scores] max_err=8.17e-05  mean_err=2.47e-05  (tol=1e-03) ✅
✅ @addrules [:t_add] — 4 règles
  [:x] max_err=1.88e-04  mean_err=6.53e-05  (tol=1e-02) ✅
  [:pred] max_err=4.01e-05  mean_err=1.50e-05  (tol=5e-03) ✅
Test Summary:                                   | Pass  Total  Time
Backward — gradient checks (tol=1e-3, eps=1e-4) |   10     10  1.2s
seed = 1234
✅ @addrules [:grad_check] — 3 règles
  [:W] max_err=2.69e-03  mean_err=5.69e-04  (tol=5e-03) ✅
  [:gamma] max_err=1.35e-0

┌ Warning: Assignment to `dev` in soft scope is ambiguous because a global variable by the same name exists: `dev` will be treated as a new local. Disambiguate by using `local dev` to suppress this warning or `global dev` to assign to the existing global variable.
└ @ c:\Users\Nevermind\Desktop\NeuroDSLF\test\test_backward.jl:310


  [:W] max_err=3.96e-04  mean_err=1.09e-04  (tol=1e-02) ✅
  ✅ M=4 N=8

--- RMSNorm CUDA (M=8, N=16) ---
✅ @addrules [:cuda_rmsnorm_8_16] — 3 règles
  [:gamma] max_err=5.97e-04  mean_err=2.92e-04  (tol=1e-02) ✅
  [:W] max_err=5.64e-04  mean_err=1.71e-04  (tol=1e-02) ✅
  ✅ M=8 N=16

--- RMSNorm CUDA (M=1, N=32) ---
✅ @addrules [:cuda_rmsnorm_1_32] — 3 règles
  [:gamma] max_err=2.56e-03  mean_err=8.12e-04  (tol=1e-02) ✅
  [:W] max_err=1.20e-03  mean_err=3.47e-04  (tol=1e-02) ✅
  ✅ M=1 N=32


In [14]:
# Test 4
include("test/test_layers.jl")

Test Summary: | Pass  Total  Time
Layers        |    6      6  0.1s


Test.DefaultTestSet("Layers", Any[Test.DefaultTestSet("LayerNorm shape", Any[], 1, false, false, true, 1.781134577953e9, 1.781134577962e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_layers.jl"), Test.DefaultTestSet("Linear shape", Any[], 1, false, false, true, 1.781134577963e9, 1.781134577963e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_layers.jl"), Test.DefaultTestSet("MultiHeadAttention shape", Any[], 1, false, false, true, 1.781134577963e9, 1.781134577982e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_layers.jl"), Test.DefaultTestSet("LlamaBlock residual shape", Any[], 1, false, false, true, 1.781134577982e9, 1.781134577983e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_layers.jl"), Test.DefaultTestSet("LlamaModel 2 layers", Any[], 2, false, false, true, 1.781134577983e9, 1.781134578069e9, false, "c:\\Users\\Nevermind\\Desktop\\NeuroDSLF\\test\\test_layers.jl")], 0, false, false, true, 1.781134577953e9, 1.7811345

In [ ]:
using CUDA
#CUDA.set_runtime_version!(v"13.2.1")  
println("=" ^ 50)
println("   NeuroDSL v4 — Test Suite")
println("=" ^ 50)
include("test/runtests.jl")

In [1]:
using Test
using LinearAlgebra
using NeuroDSL  

@testset "Validation Analytique des Gradients (Différences Finies)" begin
    # 1. Initialisation d'un petit graphe de test
    g = JuliusGraph(device=Backend.CPUDevice())
    
    H = randn(Float32, 10, 5)
    W = randn(Float32, 8, 5)
    
    set!(g, :H, H; is_param=true)
    set!(g, :W, W; is_param=true)
    
    # Y = H * W'
    addrule!(g, GraphRule(:Y, [:H, :W], :matmul; attrs=Dict(:trans_b=>true)))
    addrule!(g, GraphRule(:loss, [:Y], :sum_matrix))
    
    # --- CtxStore ---
    ctx = CtxStore()
    
    # 2. Calcul du gradient analytique via NeuroDSL
    demand!(g, :loss; ctx_store=ctx)
    backward_graph!(g, :loss; ctx_store=ctx)
    grad_W_dsl = copy(node(g, :W).gradient)
    
    # 3. Calcul par différences finies (Théorème de Taylor)
    epsilon = 1f-3 # Correction ici : 1f-3 définit correctement un Float32
    grad_W_fd = zeros(Float32, size(W))
    
    for i in eachindex(W)
        W_orig = W[i]
        
        # Perturbation + epsilon
        W[i] = W_orig + epsilon
        set!(g, :W, W; is_param=true)
        invalidate_all!(g) 
        loss_plus = demand!(g, :loss)[1]
        
        # Perturbation - epsilon
        W[i] = W_orig - epsilon
        set!(g, :W, W; is_param=true)
        invalidate_all!(g)
        loss_minus = demand!(g, :loss)[1]
        
        # Restauration
        W[i] = W_orig
        
        # Différence finie centrée
        grad_W_fd[i] = (loss_plus - loss_minus) / (2f0 * epsilon)
    end
    
    # 4. Assertion : L'erreur relative doit être minime
    @test isapprox(grad_W_dsl, grad_W_fd, rtol=1e-3)
    println("✅ Gradients matmul transposés validés ! Différence quasi nulle.")
end

✅ Gradients matmul transposés validés ! Différence quasi nulle.
Test Summary:                                            | Pass  Total  Time
Validation Analytique des Gradients (Différences Finies) |    1      1  8.2s


Test.DefaultTestSet("Validation Analytique des Gradients (Différences Finies)", Any[], 1, false, false, true, 1.781135852993e9, 1.781135861183e9, false, "In[1]")

In [32]:
using BenchmarkTools
using Printf

# 1. Setup propre
g = JuliusGraph(device=Backend.CPUDevice())
H = randn(Float32, 10, 5)
W = randn(Float32, 8, 5)
set!(g, :H, H; is_param=true)
set!(g, :W, W; is_param=true)
addrule!(g, GraphRule(:Y, [:H, :W], :matmul; attrs=Dict(:trans_b=>true)))
addrule!(g, GraphRule(:loss, [:Y], :sum_matrix))

function bench_fwd(g, ctx)
    invalidate_all!(g) 
    empty!(ctx) # AJOUTE CECI : vide le cache de résultats
    demand!(g, :loss; ctx_store=ctx)
end

function bench_bwd(g, ctx)
    invalidate_all!(g)
    empty!(ctx) # AJOUTE CECI
    demand!(g, :loss; ctx_store=ctx) 
    backward_graph!(g, :loss; ctx_store=ctx)
end

# 4. Lancement
ctx = CtxStore()

println("Lancement du Benchmark...")
b_fwd = @benchmark bench_fwd($g, $ctx)
b_bwd = @benchmark bench_bwd($g, $ctx)

display(b_fwd)
display(b_bwd)

Lancement du Benchmark...


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  9.000 μs … 44.300 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     9.400 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   9.597 μs ±  1.245 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

     ▄█ ▆▁                                                    
  ▂▃▆██▁██▆▄▃▁▃▂▂▂▂▁▂▂▂▂▂▁▂▂▂▂▂▁▂▂▂▁▁▂▂▂▁▂▁▂▁▂▂▂▁▂▂▂▂▂▁▂▂▂▂▂ ▃
  9 μs           Histogram: frequency by time        13.8 μs <

 Memory estimate: 1.67 KiB, allocs estimate: 21.

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  20.300 μs …  3.530 ms  ┊ GC (min … max): 0.00% … 97.11%
 Time  (median):     21.000 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   21.997 μs ± 35.190 μs  ┊ GC (mean ± σ):  1.56% ±  0.97%

  ▃███▅▃▁                                                     ▂
  ████████▆▆▅▆▇▅▇▅▆▆▇▆▆▅▆▆▆▇▄▆▅▆▅▆▅▆▄▆▅▅▇▆▆▅▆▅▆▆▆▆▄▆▄▆▆▇▆▆▆▆▅ █
  20.3 μs      Histogram: log(frequency) by time      35.3 μs <

 Memory estimate: 5.36 KiB, allocs estimate: 57.

In [ ]:
using Profile
# 1. On chauffe (pour compiler)
demand!(g, :loss; ctx_store=ctx) 

# 2. On profile les allocations
Profile.clear()
@profile demand!(g, :loss; ctx_store=ctx)



In [ ]:
using Profile
Profile.clear()
@profile for i in 1:100_000
    demand!(g, :loss; ctx_store=ctx)
end

# Affiche le résultat sous forme d'arbre
Profile.print(format=:tree, sortedby=:count)

In [ ]:
# Fais chauffer le moteur pour qu'il remplisse ses caches internes
for i in 1:1000
    demand!(g, :loss; ctx_store=ctx)
end
# Maintenant, le benchmark sera plus représentatif
@benchmark bench_fwd($g, $ctx)

In [ ]:
# Avant ton benchmark réel, force 10 000 passages
for _ in 1:10000
    demand!(g, :loss; ctx_store=ctx)
end
# Maintenant, le GC a probablement nettoyé ce qu'il pouvait
@btime demand!(g, :loss; ctx_store=$ctx)

# Test Relu

In [3]:
# ── TEST 1 : ReLU backward — comparaison gradient analytique vs numérique ──
using Printf
function relu_grad_check(; M=6, N=8, eps=1f-3, tol=5f-2)
    dev = NeuroDSL.Backend.CPUDevice()
    g   = NeuroDSL.JuliusGraph(namespace=:t_relu, device=dev)

    x_val = randn(Float32, M, N)
    NeuroDSL.set!(g, :x, x_val; is_param=true, namespace=:t_relu)
    NeuroDSL.set!(g, :Z, zeros(Float32, M, N);
                  atom_type=NeuroDSL.Datom, namespace=:t_relu)

    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:r, [:x], :relu;    namespace=:t_relu))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [:r, :Z], :mse_loss; namespace=:t_relu))

    # ---- gradient analytique ----
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, :L; ctx_store=ctx)
    NeuroDSL.backward_graph!(g, :L; ctx_store=ctx)
    grad_a = copy(Array(NeuroDSL.node(g, :x; namespace=:t_relu).gradient))

    # ---- gradient numérique (différences finies centrées) ----
    pn       = NeuroDSL.node(g, :x; namespace=:t_relu)
    orig_cpu = copy(Array(pn.value))
    grad_n   = zeros(Float32, size(orig_cpu))

    for i in eachindex(orig_cpu)
        for (sign, δ) in ((+1, eps), (-1, -eps))
            v = copy(orig_cpu); v[i] += δ
            pn.value = v
            NeuroDSL.invalidate_all!(g; namespace=:t_relu)
            ctx_tmp = NeuroDSL.CtxStore()
            l = sum(Array(NeuroDSL.demand!(g, :L; ctx_store=ctx_tmp, namespace=:t_relu)))
            sign == 1 ? (grad_n[i]  = l) : (grad_n[i] = (grad_n[i] - l) / (2eps))
        end
    end
    pn.value = orig_cpu; NeuroDSL.invalidate_all!(g; namespace=:t_relu)

    diff = maximum(abs.(grad_a .- grad_n))
    ok   = diff < tol
    @printf "  max|grad_analytique − grad_numérique| = %.6f  %s\n" diff (ok ? "✅" : "❌ ÉCHEC")

    # bonus : vérification directe — les positions x<0 ont gradient 0
    mask_expect = Float32.(orig_cpu .> 0f0)
    # grad_a est proportionnel à mask (via la chaîne mse_loss → relu)
    same_pattern = all((grad_a .!= 0) .== (mask_expect .!= 0))
    @printf "  Pattern zéro/non-zéro correct (x<0→0) : %s\n" (same_pattern ? "✅" : "❌")
    return ok && same_pattern
end

println("=== TEST 1 — ReLU backward ===")
relu_grad_check()

=== TEST 1 — ReLU backward ===
  max|grad_analytique − grad_numérique| = 0.000039  ✅
  Pattern zéro/non-zéro correct (x<0→0) : ✅


true

# ── TEST 2 : Dropout backward ──

In [33]:
# ── TEST 2 : Dropout backward ──
# Les finite differences ne marchent pas (mask stochastique) → on teste
# deux propriétés structurelles : identité à rate=0, et masquage à rate>0.

function dropout_grad_check(; M=8, N=6, rate=0.4f0)
    dev = NeuroDSL.Backend.CPUDevice()

    # ── 2a. rate=0 → dropout transparent, gradient = gradient sans dropout ──
    function loss_and_grad(ns, use_dropout)
        g = NeuroDSL.JuliusGraph(namespace=ns, device=dev)
        x_val = randn(Float32, M, N)   # même seed ici
        NeuroDSL.set!(g, :x, x_val; is_param=true, namespace=ns)
        NeuroDSL.set!(g, :Z, zeros(Float32, M, N); atom_type=NeuroDSL.Datom, namespace=ns)
        if use_dropout
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:d, [:x], :dropout;
                attrs=Dict{Symbol,Any}(:rate=>0f0, :training=>true), namespace=ns))
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [:d, :Z], :mse_loss; namespace=ns))
        else
            NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [:x, :Z], :mse_loss; namespace=ns))
        end
        ctx = NeuroDSL.CtxStore()
        NeuroDSL.demand!(g, :L; ctx_store=ctx, namespace=ns)
        NeuroDSL.backward_graph!(g, :L; ctx_store=ctx, namespace=ns)
        return Array(NeuroDSL.node(g, :x; namespace=ns).gradient)
    end

    # On fixe le seed pour avoir les mêmes x_val
    Random.seed!(42)
    g_nodrop = loss_and_grad(:no_drop, false)
    Random.seed!(42)
    g_drop0  = loss_and_grad(:drop0, true)

    diff_identity = maximum(abs.(g_nodrop .- g_drop0))
    ok1 = diff_identity < 1f-5
    @printf "  Dropout(rate=0) == identité — max diff : %.2e  %s\n" diff_identity (ok1 ? "✅" : "❌")

    # ── 2b. rate=0.4 → ~40% des gradients doivent être nuls ──
    g2 = NeuroDSL.JuliusGraph(namespace=:drop_rate, device=dev)
    x_val = randn(Float32, M, N)
    NeuroDSL.set!(g2, :x,  x_val;              is_param=true, namespace=:drop_rate)
    NeuroDSL.set!(g2, :Z, zeros(Float32, M, N); atom_type=NeuroDSL.Datom, namespace=:drop_rate)
    NeuroDSL.addrule!(g2, NeuroDSL.GraphRule(:d, [:x], :dropout;
        attrs=Dict{Symbol,Any}(:rate=>rate, :training=>true), namespace=:drop_rate))
    NeuroDSL.addrule!(g2, NeuroDSL.GraphRule(:L, [:d, :Z], :mse_loss; namespace=:drop_rate))

    ctx2 = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g2, :L; ctx_store=ctx2, namespace=:drop_rate)
    NeuroDSL.backward_graph!(g2, :L; ctx_store=ctx2, namespace=:drop_rate)
    grad2 = Array(NeuroDSL.node(g2, :x; namespace=:drop_rate).gradient)

    frac_zero = mean(grad2 .== 0f0)
    ok2 = abs(frac_zero - rate) < 0.25   # tolérance large (stochastique)
    @printf "  Dropout(rate=%.1f) → %.1f%% nuls (attendu ~%.0f%%)  %s\n" rate (100*frac_zero) (100*rate) (ok2 ? "✅" : "⚠️ hors tolérance")

    # ── 2c. Les positions non-masquées sont rescalées par 1/(1-rate) ──
    mask_used = (grad2 .!= 0f0)
    if any(mask_used)
        # le gradient non-nul doit être dy * 1/(1-rate) — vérifier l'échelle
        # grad_nodrop serait (2/N)*(x - 0)*1 = (2/N)*x
        # grad_drop non-nul devrait être (2/N)*x / (1-rate)
        ratio = mean(abs.(grad2[mask_used])) / mean(abs.(g_nodrop[mask_used]))
        expected_ratio = 1f0 / (1f0 - rate)
        ok3 = abs(ratio - expected_ratio) < 0.3
        @printf "  Rescaling 1/(1-rate) — ratio observé : %.3f  attendu : %.3f  %s\n" ratio expected_ratio (ok3 ? "✅" : "⚠️")
    end
    return ok1 && ok2
end

using Random, Statistics, Printf
println("\n=== TEST 2 — Dropout backward ===")
dropout_grad_check()


=== TEST 2 — Dropout backward ===
  Dropout(rate=0) == identité — max diff : 0.00e+00  ✅
  Dropout(rate=0.4) → 35.4% nuls (attendu ~40%)  ✅
  Rescaling 1/(1-rate) — ratio observé : 3.163  attendu : 1.667  ⚠️


true

In [20]:
using .NeuroDSL

In [34]:
# Colle ça dans une cellule notebook après avoir rechargé le module

using CUDA
CUDA.allowscalar(false)   # ← crashait avant, ne doit plus crasher

g = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice())
NeuroDSL.set!(g, :x,   CUDA.randn(Float32, 16, 64); namespace=:default)
NeuroDSL.set!(g, :tgt, CUDA.zeros(Float32, 16, 64); atom_type=NeuroDSL.Datom)

# Un bloc flash attention 1 tête, d_head=64
NeuroDSL._register_flash_attn_in_dispatch!()
NeuroDSL.set!(g, :Wq, CUDA.randn(Float32, 64, 64); is_param=true)
NeuroDSL.set!(g, :Wk, CUDA.randn(Float32, 64, 64); is_param=true)
NeuroDSL.set!(g, :Wv, CUDA.randn(Float32, 64, 64); is_param=true)

NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:q, [:x, :Wq], :matmul; attrs=Dict(:trans_b=>true)))
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:k, [:x, :Wk], :matmul; attrs=Dict(:trans_b=>true)))
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:v, [:x, :Wv], :matmul; attrs=Dict(:trans_b=>true)))
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:a, [:q,:k,:v], :flash_attn; attrs=Dict(:d_head=>64,:causal=>true)))
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [:a, :tgt], :mse_loss))

ctx = NeuroDSL.CtxStore()
loss = NeuroDSL.demand!(g, :L; ctx_store=ctx)
CUDA.synchronize()
println("Forward ✅  loss = ", Array(loss)[1])

NeuroDSL.backward_graph!(g, :L; ctx_store=ctx)
CUDA.synchronize()
println("Backward ✅  grad Wq norm = ",
        sqrt(sum(Array(NeuroDSL.node(g,:Wq).gradient).^2)))

✅ Op :flash_attn enregistré


┌ Warning: Shape inference non implémentée pour :flash_attn, utilisation de la forme du premier argument
└ @ Main.NeuroDSL c:\Users\Nevermind\Desktop\NeuroDSLF\src\dispatch.jl:67


Forward ✅  loss = 53.437473
Backward ✅  grad Wq norm = 44.916603


# memory 

In [ ]:
using CUDA

In [45]:
function measure_peak_memory(f)
    CUDA.reclaim()  # libère la mémoire non utilisée
    CUDA.unsafe_reset_peak_memory()  # si disponible, sinon on utilise @time
    # Alternative: utiliser CUDA.@time qui donne le pic
    t, bytes, gbytes, pool_alloc, pool_free, peak = CUDA.@time f()
    return peak
end

measure_peak_memory (generic function with 1 method)

In [46]:
using  CUDA

# Création d'un graphe profond
function create_deep_graph(g::JuliusGraph, ns::Symbol, depth::Int, dim::Int)
    x = Symbol(:x)
    set!(g, x, CUDA.randn(Float32, 128, dim); is_param=false, namespace=ns)
    prev = x
    for i in 1:depth
        w = Symbol(:W, i)
        set!(g, w, CUDA.randn(Float32, dim, dim) .* 0.01f0; is_param=true, namespace=ns)
        mul = Symbol(:mul, i)
        addrule!(g, GraphRule(mul, [prev, w], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
        act = Symbol(:act, i)
        addrule!(g, GraphRule(act, [mul], :relu; namespace=ns))
        prev = act
    end
    loss_sym = Symbol(:loss)
    set!(g, :target, CUDA.zeros(Float32, 128, dim); atom_type=Datom, namespace=ns)
    addrule!(g, GraphRule(loss_sym, [prev, :target], :mse_loss; namespace=ns))
    return loss_sym
end

function run_without_release(g, loss_sym, ns)
    ctx = CtxStore()
    demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns)
end

function run_with_release(g, loss_sym, ns, keep_every=4)
    ctx = CtxStore()
    order = topo_order!(g; namespace=ns)
    # Forward en libérant certaines activations
    for (i, sym) in enumerate(order)
        haskey(g.rules[ns], sym) || continue
        # Exécute la règle (en utilisant l'API publique)
        demand!(g, sym; ctx_store=ctx, namespace=ns)
        # Libère les activations qui ne sont pas des points de contrôle
        if i % keep_every != 0 && sym != loss_sym
            nd = g.nodes[ns][sym]
            nd.value = nothing
            nd.valid = false
        end
    end
    # Le backward recompute automatiquement les valeurs manquantes via demand!
    backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns)
end

function memory_benchmark(depth=5, dim=512, keep_every=4)
    ns = :test
    g = JuliusGraph(namespace=ns, device=Backend.CUDADevice())
    loss_sym = create_deep_graph(g, ns, depth, dim)
    topo_order!(g; namespace=ns)

    # Échauffement
    run_without_release(g, loss_sym, ns)
    invalidate_all!(g; namespace=ns)
    zero_grads!(g; namespace=ns)
    CUDA.reclaim()

    println("--- Sans libération (conserve toutes les activations) ---")
    CUDA.@time run_without_release(g, loss_sym, ns)

    invalidate_all!(g; namespace=ns)
    zero_grads!(g; namespace=ns)
    CUDA.reclaim()

    println("\n--- Avec libération (1 activation sur $keep_every conservée) ---")
    CUDA.@time run_with_release(g, loss_sym, ns, keep_every)
end

memory_benchmark(5, 512, 2);

--- Sans libération (conserve toutes les activations) ---
  0.035922 seconds (4.28 k CPU allocations: 136.125 KiB) (72 GPU allocations: 17.251 MiB, 1.29% memmgmt time)

--- Avec libération (1 activation sur 2 conservée) ---
  0.035177 seconds (4.91 k CPU allocations: 156.062 KiB) (86 GPU allocations: 19.251 MiB, 1.62% memmgmt time)


In [47]:
using Printf

function memory_benchmark_v2(depth=5, dim=512; every=2)
    ns = :bench
    g  = NeuroDSL.JuliusGraph(namespace=ns, device=NeuroDSL.Backend.CUDADevice())
    loss_sym = create_deep_graph(g, ns, depth, dim)
    NeuroDSL.topo_order!(g; namespace=ns)

    # ── 1. MemoryPlan SANS checkpoint (forward classique) ──
    plan_base, _ = NeuroDSL.plan_memory!(g; namespace=ns)
    # Calcul du pic : nombre de slots × taille d'un tenseur (Float32, batch=32, dim)
    # Tous les tenseurs intermédiaires ont la même forme (32, dim) sauf les paramètres
    tensor_bytes = 32 * dim * sizeof(Float32)
    peak_base_MiB = plan_base.n_slots * tensor_bytes / 1024^2

    # ── 2. MemoryPlan AVEC checkpoint (forward + recomputation) ──
    # On construit manuellement l'empreinte mémoire du checkpointing :
    # - checkpoints sauvegardés : ensemble des nœuds checkpointés
    # - recomputation : pic dû au segment le plus long entre deux checkpoints
    cd  = NeuroDSL.CheckpointData(every=every)
    sch = NeuroDSL.CheckpointSchedule(g, cd; namespace=ns)

    # Nœuds checkpointés (ceux qui restent en mémoire)
    n_checkpoints = length(sch.checkpoints)
    # Pic de recomputation : longueur max d'un segment sans checkpoint
    order = sch.order
    checkpoint_positions = [i for (i, sym) in enumerate(order) if sym in sch.checkpoints]
    if isempty(checkpoint_positions)
        max_segment = length(order)
    else
        # Ajouter le début et la fin pour couvrir tout l'ordre
        positions = [0; checkpoint_positions; length(order)+1]
        max_segment = maximum(diff(positions))
    end

    # Pic total = checkpoints + segment le plus long
    peak_ckpt_slots = n_checkpoints + max_segment - 1   # -1 car les checkpoints sont inclus dans les segments
    peak_ckpt_MiB = peak_ckpt_slots * tensor_bytes / 1024^2

    println("=== depth=$depth  dim=$dim  every=$every ===")
    println("  Taille d'un tenseur : $(round(tensor_bytes/1024^2, digits=3)) MiB")
    @printf "  Baseline    : %d slots → %.2f MiB\n" plan_base.n_slots peak_base_MiB
    @printf "  Checkpoint  : %d slots → %.2f MiB\n" peak_ckpt_slots peak_ckpt_MiB
    @printf "  Économie    : %.2f MiB (%.1f%%)\n" (peak_base_MiB - peak_ckpt_MiB) (100*(peak_base_MiB - peak_ckpt_MiB)/peak_base_MiB)
end

memory_benchmark_v2 (generic function with 3 methods)

In [48]:
memory_benchmark_v2(10, 1024; every=3)
memory_benchmark_v2(20, 2048; every=4)

=== depth=10  dim=1024  every=3 ===
  Taille d'un tenseur : 0.125 MiB

[ Info: CheckpointSchedule: 20 checkpoints, 13 recomputables



  Baseline    : 33 slots → 4.12 MiB
  Checkpoint  : 22 slots → 2.75 MiB
  Économie    : 1.38 MiB (33.3%)
=== depth=20  dim=2048  every=4 ===
  Taille d'un tenseur : 0.25 MiB
  Baseline    : 63 slots → 15.75 MiB
  Checkpoint  : 36 slots → 9.00 MiB
  Économie    : 6.75 MiB (42.9%)


[ Info: CheckpointSchedule: 33 checkpoints, 30 recomputables


In [36]:


function create_deep_graph(g::JuliusGraph, ns::Symbol, depth::Int, dim::Int, batch_size::Int)
    x = Symbol(:x)
    set!(g, x, CUDA.randn(Float32, batch_size, dim); is_param=false, namespace=ns)
    prev = x
    for i in 1:depth
        w = Symbol(:W, i)
        set!(g, w, CUDA.randn(Float32, dim, dim) .* 0.01f0; is_param=true, namespace=ns)
        mul = Symbol(:mul, i)
        addrule!(g, GraphRule(mul, [prev, w], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
        act = Symbol(:act, i)
        addrule!(g, GraphRule(act, [mul], :relu; namespace=ns))
        prev = act
    end
    loss_sym = Symbol(:loss)
    set!(g, :target, CUDA.zeros(Float32, batch_size, dim); atom_type=Datom, namespace=ns)
    addrule!(g, GraphRule(loss_sym, [prev, :target], :mse_loss; namespace=ns))
    return loss_sym
end

function run_normal(g, loss_sym, ns)
    ctx = CtxStore()
    demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns)
end

function run_with_release(g, loss_sym, ns, keep_every=2)
    ctx = CtxStore()
    order = topo_order!(g; namespace=ns)
    for (i, sym) in enumerate(order)
        haskey(g.rules[ns], sym) || continue
        demand!(g, sym; ctx_store=ctx, namespace=ns)
        if i % keep_every != 0 && sym != loss_sym
            nd = g.nodes[ns][sym]
            nd.value = nothing
            nd.valid = false
        end
    end
    backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns)
end

function memory_benchmark(depth=50, dim=256, keep_every=2, batch_size=128)
    ns = :test
    g = JuliusGraph(namespace=ns, device=Backend.CUDADevice())
    println("📐 Création du graphe avec $depth couches, dim=$dim, batch=$batch_size...")
    loss_sym = create_deep_graph(g, ns, depth, dim, batch_size)
    topo_order!(g; namespace=ns)

    # Échauffement
    run_normal(g, loss_sym, ns)
    invalidate_all!(g; namespace=ns)
    zero_grads!(g; namespace=ns)
    CUDA.reclaim()

    println("\n--- Exécution normale (conserve toutes les activations) ---")
    CUDA.@time run_normal(g, loss_sym, ns)

    invalidate_all!(g; namespace=ns)
    zero_grads!(g; namespace=ns)
    CUDA.reclaim()

    println("\n--- Exécution avec libération (1 activation sur $keep_every conservée) ---")
    CUDA.@time run_with_release(g, loss_sym, ns, keep_every)
end

memory_benchmark (generic function with 5 methods)

In [39]:
memory_benchmark(10, 4096, 2, 256);

📐 Création du graphe avec 10 couches, dim=4096, batch=256...

--- Exécution normale (conserve toutes les activations) ---
  0.059085 seconds (7.99 k CPU allocations: 252.172 KiB) (137 GPU allocations: 1.461 GiB, 18.25% memmgmt time)

--- Exécution avec libération (1 activation sur 2 conservée) ---
  0.036619 seconds (9.21 k CPU allocations: 296.469 KiB) (161 GPU allocations: 1.516 GiB, 27.02% memmgmt time)


In [40]:
memory_benchmark(100, 128, 2, 1024) ;  # depth=100, dim=128, keep_every=2, batch=1024;

📐 Création du graphe avec 100 couches, dim=128, batch=1024...

--- Exécution normale (conserve toutes les activations) ---
  0.068856 seconds (74.24 k CPU allocations: 2.295 MiB) (1.31 k GPU allocations: 264.503 MiB, 10.42% memmgmt time)

--- Exécution avec libération (1 activation sur 2 conservée) ---
  0.085089 seconds (86.73 k CPU allocations: 2.689 MiB) (1.56 k GPU allocations: 340.003 MiB, 11.24% memmgmt time)


In [41]:
# Profondeur 200, dim=64, batch=2048, keep_every=2
memory_benchmark(200, 64, 2, 2048);

📐 Création du graphe avec 200 couches, dim=64, batch=2048...

--- Exécution normale (conserve toutes les activations) ---
  0.125979 seconds (147.83 k CPU allocations: 4.548 MiB) (2.61 k GPU allocations: 508.255 MiB, 11.12% memmgmt time)

--- Exécution avec libération (1 activation sur 2 conservée) ---
  0.164843 seconds (172.83 k CPU allocations: 5.337 MiB) (3.11 k GPU allocations: 658.256 MiB, 10.19% memmgmt time)


In [42]:
function peak_memory_test(depth, dim, batch, keep_every)
    ns = :test
    g = JuliusGraph(namespace=ns, device=Backend.CUDADevice())
    loss_sym = create_deep_graph(g, ns, depth, dim, batch)
    topo_order!(g; namespace=ns)

    # Échauffement
    run_normal(g, loss_sym, ns)
    invalidate_all!(g; namespace=ns); zero_grads!(g; namespace=ns)
    CUDA.reclaim()
    CUDA.synchronize()

    total_mem = CUDA.total_memory()
    free_mem  = CUDA.free_memory()
    initial   = total_mem - free_mem

    if keep_every == 0
        run_normal(g, loss_sym, ns)
    else
        run_with_release(g, loss_sym, ns, keep_every)
    end

    CUDA.synchronize()
    total_mem = CUDA.total_memory()
    free_mem  = CUDA.free_memory()
    used_after = total_mem - free_mem
    peak = used_after - initial
    return peak
end

# Comparaison
peak_normal = peak_memory_test(10, 4096, 256, 0)
peak_release = peak_memory_test(10, 4096, 256, 2)
println("Pic mémoire normal : $(peak_normal / 1e6) Mo")
println("Pic mémoire release: $(peak_release / 1e6) Mo")

Pic mémoire normal : 1577.058304 Mo
Pic mémoire release: 1375.731712 Mo


# adds

In [7]:
using .NeuroDSL

g = NeuroDSL.JuliusGraph()
NeuroDSL.set!(g, :x, ones(Float32, 2,2))
NeuroDSL.set!(g, :y, zeros(Float32, 2,2))

NeuroDSL._watch!(g, :y, :x)
NeuroDSL.node(g, :y).on_change = (graph, sym, ns) -> println("$sym a été invalidé !")

# Ceci doit déclencher le callback
NeuroDSL._invalidate_downstream!(g, :x, :default)

y a été invalidé !


In [8]:
g = NeuroDSL.JuliusGraph()
NeuroDSL.set!(g, :x, ones(Float32, 2,2))
NeuroDSL.set!(g, :y, zeros(Float32, 2,2))

NeuroDSL._watch!(g, :y, :x)
NeuroDSL.node(g, :y).on_change = (graph, sym, ns) -> println("$sym a été invalidé !")

# Invalider x sans le recréer → le watcher fonctionne
NeuroDSL._invalidate_downstream!(g, :x, :default)

y a été invalidé !


In [11]:
NeuroDSL.set!(g, :x, randn(Float32, 2,2))  # callback déclenché !

y a été invalidé !


Main.NeuroDSL.JuliusGraph(Dict{Symbol, Dict{Symbol, Main.NeuroDSL.GraphNode{Float32}}}(:default => Dict(:y => Main.NeuroDSL.GraphNode{Float32}(:y, Float32[0.0 0.0; 0.0 0.0], nothing, false, Main.NeuroDSL.Quantom, false, :default, Dict{Symbol, Any}(), Symbol[], var"#11#12"()), :x => Main.NeuroDSL.GraphNode{Float32}(:x, Float32[-0.67887396 1.2498932; -0.97599804 -1.6151873], nothing, false, Main.NeuroDSL.Quantom, false, :default, Dict{Symbol, Any}(), [:y], nothing))), Dict{Symbol, Dict{Symbol, Main.NeuroDSL.GraphRule}}(:default => Dict()), Dict{Symbol, Function}(), Dict{Symbol, Union{Nothing, Vector{Symbol}}}(:default => nothing), :default, Main.NeuroDSL.Backend.CUDADevice())

In [12]:
using BenchmarkTools, CUDA

function bench_fusion_gpu()
    dev = NeuroDSL.Backend.CUDADevice()

    g1 = NeuroDSL.JuliusGraph(device=dev)
    NeuroDSL.set!(g1, :x, CUDA.randn(Float32, 1024, 512))
    NeuroDSL.set!(g1, :w1, CUDA.randn(Float32, 512, 512))
    NeuroDSL.set!(g1, :w2, CUDA.randn(Float32, 512, 512))
    NeuroDSL.addrule!(g1, GraphRule(:h1, [:x, :w1], :matmul, attrs=Dict(:trans_b=>true)))
    NeuroDSL.addrule!(g1, GraphRule(:a1, [:h1], :relu))
    NeuroDSL.addrule!(g1, GraphRule(:h2, [:a1, :w2], :matmul, attrs=Dict(:trans_b=>true)))

    g2 = NeuroDSL.JuliusGraph(device=dev)
    NeuroDSL.set!(g2, :x, CUDA.randn(Float32, 1024, 512))
    NeuroDSL.set!(g2, :w1, CUDA.randn(Float32, 512, 512))
    NeuroDSL.set!(g2, :w2, CUDA.randn(Float32, 512, 512))
    NeuroDSL.addrule!(g2, GraphRule(:h1, [:x, :w1], :matmul, attrs=Dict(:trans_b=>true)))
    NeuroDSL.addrule!(g2, GraphRule(:a1, [:h1], :relu))
    NeuroDSL.addrule!(g2, GraphRule(:h2, [:a1, :w2], :matmul, attrs=Dict(:trans_b=>true)))
    NeuroDSL._fuse!(g2, [:h1, :a1])

    # Warm-up CUDA
    NeuroDSL.invalidate_all!(g1); NeuroDSL.demand!(g1, :h2)
    NeuroDSL.invalidate_all!(g2); NeuroDSL.demand!(g2, :h2)
    CUDA.synchronize()

    t_before = @belapsed begin
        NeuroDSL.invalidate_all!($g1)
        NeuroDSL.demand!($g1, :h2)
    end
    t_after = @belapsed begin
        NeuroDSL.invalidate_all!($g2)
        NeuroDSL.demand!($g2, :h2)
    end
    println("Avant fusion : ", round(t_before * 1e6, digits=1), " μs")
    println("Après fusion : ", round(t_after * 1e6, digits=1), " μs")
    println("Gain : ", round((t_before - t_after) / t_before * 100, digits=1), "%")
end

bench_fusion_gpu()

Avant fusion : 216.7 μs
Après fusion : 131.5 μs
Gain : 39.3%


In [16]:
using .NeuroDSL, BenchmarkTools

# Construction des deux graphes (identique à bench_fusion)
g1 = NeuroDSL.JuliusGraph()
NeuroDSL.set!(g1, :x, randn(Float32, 1024, 512))
NeuroDSL.set!(g1, :w1, randn(Float32, 512, 512))
NeuroDSL.set!(g1, :w2, randn(Float32, 512, 512))
NeuroDSL.addrule!(g1, GraphRule(:h1, [:x, :w1], :matmul, attrs=Dict(:trans_b=>true)))
NeuroDSL.addrule!(g1, GraphRule(:a1, [:h1], :relu))
NeuroDSL.addrule!(g1, GraphRule(:h2, [:a1, :w2], :matmul, attrs=Dict(:trans_b=>true)))

g2 = NeuroDSL.JuliusGraph()
NeuroDSL.set!(g2, :x, randn(Float32, 1024, 512))
NeuroDSL.set!(g2, :w1, randn(Float32, 512, 512))
NeuroDSL.set!(g2, :w2, randn(Float32, 512, 512))
NeuroDSL.addrule!(g2, GraphRule(:h1, [:x, :w1], :matmul, attrs=Dict(:trans_b=>true)))
NeuroDSL.addrule!(g2, GraphRule(:a1, [:h1], :relu))
NeuroDSL.addrule!(g2, GraphRule(:h2, [:a1, :w2], :matmul, attrs=Dict(:trans_b=>true)))
NeuroDSL._fuse!(g2, [:h1, :a1])  # fusion matmul+relu

# Maintenant le benchmark fonctionne
t_before = @benchmark begin
    NeuroDSL.invalidate_all!($g1)
    NeuroDSL.demand!($g1, :h2)
end

t_after = @benchmark begin
    NeuroDSL.invalidate_all!($g2)
    NeuroDSL.demand!($g2, :h2)
end

println("Médiane avant fusion : ", round(median(t_before).time * 1e-3, digits=1), " μs")
println("Médiane après fusion : ", round(median(t_after).time * 1e-3, digits=1), " μs")
gain = (median(t_before).time - median(t_after).time) / median(t_before).time * 100
println("Gain médian : ", round(gain, digits=1), "%")

Médiane avant fusion : 261.6 μs
Médiane après fusion : 164.4 μs
Gain médian : 37.2%


In [4]:
using .NeuroDSL, BenchmarkTools, Random

function test_upstream_invalidation_working()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    D = 64
    NeuroDSL.set!(g, :x, randn(Float32, 128, D))
    NeuroDSL.set!(g, :y, zeros(Float32, 128, D); atom_type=NeuroDSL.Datom)

    prev = :x
    params_list = Symbol[]
    for i in 1:10
        w_sym = Symbol(:W, i)
        NeuroDSL.set!(g, w_sym, randn(Float32, D, D) .* 0.01f0; is_param=true)
        push!(params_list, w_sym)
        out_sym = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out_sym
    end
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss))

    w10 = params_list[end]
    new_W10 = randn(Float32, D, D) .* 0.01f0

    # Warm-up et premier backward complet
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym)

    # --- Méthode NAÏVE : on réinitialise tout avant chaque mesure ---
    t_naive = @belapsed begin
        NeuroDSL.zero_grads!($g)
        NeuroDSL.invalidate_all!($g)   # forward aussi réinitialisé
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym)
    end

    # --- Méthode INCRÉMENTALE : on ne change que W10 ---
    t_incr = @belapsed begin
        NeuroDSL.set!($g, $w10, $new_W10; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)   # forward : rien à recalculer (sauf loss)
        NeuroDSL.backward_graph!($g, $loss_sym)  # seules les couches affectées recalculent
    end

    println("Temps naïf (tout recalculé)      : ", round(t_naive * 1e6, digits=1), " μs")
    println("Temps incrémental (partiel)       : ", round(t_incr * 1e6, digits=1), " μs")
    println("Gain : ", round((t_naive - t_incr) / t_naive * 100, digits=1), "%")
end

test_upstream_invalidation_working()

Temps naïf (tout recalculé)      : 38496.7 μs
Temps incrémental (partiel)       : 27598.1 μs
Gain : 28.3%


In [5]:
function bench_incr_stable()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    D = 64
    NeuroDSL.set!(g, :x, randn(Float32, 128, D))
    NeuroDSL.set!(g, :y, zeros(Float32, 128, D); atom_type=NeuroDSL.Datom)

    prev = :x
    params_list = Symbol[]
    for i in 1:10
        w_sym = Symbol(:W, i)
        NeuroDSL.set!(g, w_sym, randn(Float32, D, D) .* 0.01f0; is_param=true)
        push!(params_list, w_sym)
        out_sym = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out_sym
    end
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss))

    w10 = params_list[end]
    new_W10 = randn(Float32, D, D) .* 0.01f0

    # 1. Warm-up: Get the graph into a "Valid" state
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym; full=true)

    # ─── FAIR BENCHMARK STRATEGY ───
    # We want to measure: "Given the graph is valid, how long does it take 
    # to change W10 and perform the backward pass?"
    
    # We use a function to wrap the action so we can benchmark it
    function run_pass(g, loss, w_sym, val, is_full)
        # This is the actual work we are measuring
        NeuroDSL.set!(g, w_sym, val; is_param=true)
        NeuroDSL.demand!(g, loss)
        NeuroDSL.backward_graph!(g, loss; full=is_full)
    end

    # IMPORTANT: Since the benchmark runs many times, the first call makes it dirty.
    # To fix this, we MUST restore the state inside the benchmark, but 
    # we subtract the restoration cost, OR we use a different approach.
    
    # BEST APPROACH: Compare the two modes on a large enough scale 
    # where the "Full" pass cost is dominant.
    
    println("Measuring Naive (Full) Pass...")
    t_naive = @benchmark begin
        # We force a full re-computation of everything
        # To make it fair, we simulate the update
        NeuroDSL.set!($g, $w10, $new_W10; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=true)
    end

    # Now we reset to valid state for the incremental test
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym; full=true)

    println("Measuring Incremental Pass...")
    t_incr = @benchmark begin
        # We perform the same update, but use the incremental engine
        NeuroDSL.set!($g, $w10, $new_W10; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=false)
    end

    med_naive = median(t_naive).time * 1e-3
    med_incr = median(t_incr).time * 1e-3
    gain = (med_naive - med_incr) / med_naive * 100
    
    println("\nResults:")
    println("Médiane naïve (Full)       : ", round(med_naive, digits=1), " μs")
    println("Médiane incrémentale (Surgical) : ", round(med_incr, digits=1), " μs")
    println("Gain : ", round(gain, digits=1), "%")
end

bench_incr_stable()

Measuring Naive (Full) Pass...
Measuring Incremental Pass...

Results:
Médiane naïve (Full)       : 39534.2 μs
Médiane incrémentale (Surgical) : 36981.2 μs
Gain : 6.5%


In [22]:
function bench_incr_stress_test()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    # 🚀 SCALE UP: Larger dimensions and more layers
    D = 512             # Increased from 64
    depth = 50          # Increased from 10
    batch = 128         # Fixed batch size
    
    NeuroDSL.set!(g, :x, randn(Float32, batch, D))
    NeuroDSL.set!(g, :y, zeros(Float32, batch, D); atom_type=NeuroDSL.Datom)

    prev = :x
    params_list = Symbol[]
    for i in 1:depth
        w_sym = Symbol(:W, i)
        NeuroDSL.set!(g, w_sym, randn(Float32, D, D) .* 0.01f0; is_param=true)
        push!(params_list, w_sym)
        out_sym = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out_sym
    end
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss))

    # We modify the LAST parameter (W50). 
    # In Naive mode, we recompute 50 layers.
    # In Incremental mode, we only recompute the a few nodes near the loss.
    w_last = params_list[end]
    new_W_last = randn(Float32, D, D) .* 0.01f0

    # Warm-up
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym; full=true)

    println("Running Stress Test: Depth=$depth, Dim=$D...")

    # Measure Naive
    t_naive = @benchmark begin
        NeuroDSL.set!($g, $w_last, $new_W_last; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=true)
    end

    # Reset to valid for incremental
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym; full=true)

    # Measure Incremental
    t_incr = @benchmark begin
        NeuroDSL.set!($g, $w_last, $new_W_last; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=false)
    end

    med_naive = median(t_naive).time * 1e-3
    med_incr = median(t_incr).time * 1e-3
    gain = (med_naive - med_incr) / med_naive * 100
    
    println("\nResults:")
    println("Médiane naïve (Full)       : ", round(med_naive, digits=1), " μs")
    println("Médiane incrémentale (Surgical) : ", round(med_incr, digits=1), " μs")
    println("Gain : ", round(gain, digits=1), "%")
end

bench_incr_stress_test()

Running Stress Test: Depth=50, Dim=512...

Results:
Médiane naïve (Full)       : 2.5515427e6 μs
Médiane incrémentale (Surgical) : 2.5147011e6 μs
Gain : 1.4%


In [20]:
function test_upstream_invalidation_STRESS()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    D = 512          # 🚀 INCREASED (from 64)
    depth = 50       # 🚀 INCREASED (from 10)
    
    NeuroDSL.set!(g, :x, randn(Float32, 128, D))
    NeuroDSL.set!(g, :y, zeros(Float32, 128, D); atom_type=NeuroDSL.Datom)

    prev = :x
    params_list = Symbol[]
    for i in 1:depth
        w_sym = Symbol(:W, i)
        NeuroDSL.set!(g, w_sym, randn(Float32, D, D) .* 0.01f0; is_param=true)
        push!(params_list, w_sym)
        out_sym = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out_sym
    end
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss))

    w_last = params_list[end]
    new_W_last = randn(Float32, D, D) .* 0.01f0

    # Warm-up
    NeuroDSL.demand!(g, loss_sym)
    NeuroDSL.backward_graph!(g, loss_sym; full=true)

    println("Running Stress Test: Depth=$depth, Dim=$D...")

    t_naive = @belapsed begin
        NeuroDSL.zero_grads!($g)
        NeuroDSL.invalidate_all!($g)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=true)
    end

    t_incr = @belapsed begin
        NeuroDSL.set!($g, $w_last, $new_W_last; is_param=true)
        NeuroDSL.demand!($g, $loss_sym)
        NeuroDSL.backward_graph!($g, $loss_sym; full=false)
    end

    println("Temps naïf (Full)      : ", round(t_naive * 1e6, digits=1), " μs")
    println("Temps incrémental (Surgical) : ", round(t_incr * 1e6, digits=1), " μs")
    println("Gain : ", round((t_naive - t_incr) / t_naive * 100, digits=1), "%")
end

test_upstream_invalidation_STRESS()

Running Stress Test: Depth=50, Dim=512...
Temps naïf (Full)      : 2.6517284e6 μs
Temps incrémental (Surgical) : 2.5114392e6 μs
Gain : 5.3%


In [19]:
using .NeuroDSL, BenchmarkTools, Random


function run_naive(g)
    NeuroDSL.invalidate_all!(g)
    NeuroDSL.zero_grads!(g)
    NeuroDSL.demand!(g, :loss)
    NeuroDSL.backward_graph!(g, :loss; full=true)
end

function run_incremental(g)
    # We assume the 'set!' happened already
    NeuroDSL.demand!(g, :loss)
    NeuroDSL.backward_graph!(g, :loss; full=false)
end

# ── 2. The main benchmark function ──────────────────────────────────────
function bench_branching_innovation()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    D = 512
    NeuroDSL.set!(g, :x, randn(Float32, 128, D))
    
    # PATH A
    NeuroDSL.set!(g, :Wa, randn(Float32, D, D) .* 0.01f0; is_param=true)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:out_a, [:x, :Wa], :matmul; attrs=Dict(:trans_b=>true)))
    
    # PATH B
    NeuroDSL.set!(g, :Wb, randn(Float32, D, D) .* 0.01f0; is_param=true)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:out_b, [:x, :Wb], :matmul; attrs=Dict(:trans_b=>true)))
    
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:sum, [:out_a, :out_b], :add))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [:sum], :sum_matrix))

    # Warm-up
    NeuroDSL.demand!(g, :loss)
    NeuroDSL.backward_graph!(g, :loss; full=true)

    new_Wa = randn(Float32, D, D) .* 0.01f0

    println("Measuring Naive Execution...")
    # We call the global function and pass the graph
    t_naive = @benchmark run_naive($g)

    # Reset state for incremental
    NeuroDSL.demand!(g, :loss)
    NeuroDSL.backward_graph!(g, :loss; full=true)
    
    # Trigger the change that makes it incremental
    NeuroDSL.set!(g, :Wa, new_Wa; is_param=true)

    println("Measuring Incremental Execution...")
    t_incr = @benchmark run_incremental($g)

    med_naive = median(t_naive).time * 1e-3
    med_incr = median(t_incr).time * 1e-3
    gain = (med_naive - med_incr) / med_naive * 100
    
    println("\nResults (Execution only):")
    println("Médiane naïve (Full)       : ", round(med_naive, digits=1), " μs")
    println("Médiane incrémentale (Surgical) : ", round(med_incr, digits=1), " μs")
    println("Gain : ", round(gain, digits=1), "%")
end

# Now run it
bench_branching_innovation()


Measuring Naive Execution...
Measuring Incremental Execution...

Results (Execution only):
Médiane naïve (Full)       : 16662.1 μs
Médiane incrémentale (Surgical) : 12518.2 μs
Gain : 24.9%


# -----

In [11]:
using .NeuroDSL

function check_invalidation()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)
    D = 64
    NeuroDSL.set!(g, :x, randn(Float32, 128, D))
    NeuroDSL.set!(g, :y, zeros(Float32, 128, D); atom_type=NeuroDSL.Datom)

    prev = :x
    params_list = Symbol[]
    for i in 1:10
        w_sym = Symbol(:W, i)
        NeuroDSL.set!(g, w_sym, randn(Float32, D, D) .* 0.01f0; is_param=true)
        push!(params_list, w_sym)
        out_sym = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_sym, [prev, w_sym], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out_sym
    end
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :y], :mse_loss))

    # Forward+backward initiaux
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx)
    NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx)

    # Modifier W10
    w10 = params_list[end]
    new_W10 = randn(Float32, D, D) .* 0.01f0
    NeuroDSL.set!(g, w10, new_W10; is_param=true)

    # Afficher l'état valid des nœuds
    println("État après modification de W10 :")
    for sym in [:x, :y, :W1, :h1, :W5, :h5, :W9, :h9, :W10, :h10, :loss]
        nd = get(g.nodes[:default], sym, nothing)
        if nd !== nothing
            println("  $sym : valid=$(nd.valid), gradient=$(nd.gradient === nothing ? "nothing" : "présent")")
        end
    end
end

check_invalidation()

État après modification de W10 :
  x : valid=false, gradient=nothing
  y : valid=false, gradient=nothing
  W1 : valid=false, gradient=nothing
  h1 : valid=true, gradient=nothing
  W5 : valid=false, gradient=nothing
  h5 : valid=true, gradient=nothing
  W9 : valid=false, gradient=nothing
  h9 : valid=true, gradient=nothing
  W10 : valid=false, gradient=nothing
  h10 : valid=false, gradient=nothing
  loss : valid=false, gradient=nothing


In [12]:
using .NeuroDSL

g = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CPUDevice())
NeuroDSL.set!(g, :x, ones(Float32, 2,2))
NeuroDSL.set!(g, :y, zeros(Float32, 2,2))
NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [:x, :y], :mse_loss))

# Watcher
NeuroDSL.node(g, :loss).on_change = (graph, sym, ns) -> begin
    val = graph.nodes[ns][sym].value
    if val !== nothing && val[1] < 0.1f0
        println("⏹ Early stopping triggered! loss = ", val[1])
    end
end

# Mettre la loss à une valeur très basse
NeuroDSL.node(g, :x).value .= 0.01f0
NeuroDSL.invalidate_all!(g)
NeuroDSL.demand!(g, :loss)

1-element Vector{Float32}:
 0.0001

In [13]:
function test_watch_earlystopping()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)
    NeuroDSL.set!(g, :x, randn(Float32, 128, 64))
    NeuroDSL.set!(g, :y, randn(Float32, 128, 64); atom_type=NeuroDSL.Datom)
    h1 = NeuroDSL.Linear(64, 32)(g, :x, :fc1)
    h2 = NeuroDSL.Linear(32, 64)(g, h1, :fc2)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [h2, :y], :mse_loss))

    # Watcher : arrêter l'entraînement si loss < 0.001
    stopped = Ref(false)
    NeuroDSL.node(g, :loss).on_change = (graph, sym, ns) -> begin
        val = graph.nodes[ns][sym].value
        if val !== nothing && val[1] < 0.001f0
            stopped[] = true
            println("⏹ Early stopping déclenché ! loss = ", val[1])
        end
    end

    # Boucle d'entraînement (normalement AdamW, ici on simule en modifiant les poids)
    for step in 1:1000
        # Avancer d'un pas d'optimisation (pour le test, on modifie juste un peu le premier paramètre)
        p = NeuroDSL.params(g)[1]
        NeuroDSL.set!(g, p.name, p.value .+ 0.01f0 .* randn(Float32, size(p.value)); is_param=true)
        
        # Recalculer la loss (le callback est déclenché si la loss passe sous le seuil)
        NeuroDSL.invalidate_all!(g)
        NeuroDSL.demand!(g, :loss)
        
        if stopped[]
            println("Arrêt à l'étape $step")
            break
        end
    end
    println("Test terminé")
end

test_watch_earlystopping()

Test terminé


In [5]:
using .NeuroDSL, Random

function test_watch_diagnostic()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    Random.seed!(123)
    X = randn(Float32, 100, 4)
    Y = 2.0f0 .* X[:, 1:1] .+ 0.1f0 .* randn(Float32, 100, 1)

    NeuroDSL.set!(g, :x, X)
    NeuroDSL.set!(g, :y, Y; atom_type=NeuroDSL.Datom)

    out = NeuroDSL.Linear(4, 1, bias=false)(g, :x, :fc)
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [out, :y], :mse_loss))

    ps = NeuroDSL.params(g)
    m1 = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    m2 = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    lr = 0.5f0
    b1, b2, eps_v, wd, clip = 0.9f0, 0.999f0, 1f-8, 0.0f0, 1f0

    ctx = NeuroDSL.CtxStore()
    for t in 1:200
        NeuroDSL.invalidate_all!(g)
        NeuroDSL.demand!(g, loss_sym; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx)

        # Afficher la loss toutes les 20 itérations
        if t % 20 == 0 || t == 1
            loss_val = NeuroDSL.node(g, loss_sym).value[1]
            println("Étape $t : loss = $loss_val")
        end

        for (i, p) in enumerate(ps)
            NeuroDSL.adamw_step!(dev, p.value, p.gradient,
                                 m1[i], m2[i], lr, b1, b2, eps_v, t, clip, wd)
        end
    end
end

test_watch_diagnostic()

Étape 1 : loss = 4.183966
Étape 20 : loss = 0.018621244
Étape 40 : loss = 0.029332414
Étape 60 : loss = 0.012043392
Étape 80 : loss = 0.009193854
Étape 100 : loss = 0.008982623
Étape 120 : loss = 0.008977173
Étape 140 : loss = 0.008974097
Étape 160 : loss = 0.008973317
Étape 180 : loss = 0.008973239
Étape 200 : loss = 0.00897323


In [7]:
using .NeuroDSL

g = NeuroDSL.JuliusGraph()
NeuroDSL.set!(g, :x, ones(Float32, 2,2))
NeuroDSL.set!(g, :y, zeros(Float32, 2,2))
NeuroDSL._watch!(g, :y, :x)   # y observe x
NeuroDSL.node(g, :y).on_change = (graph, sym, ns) -> println("$sym réagit au changement de :x")

# Modifier :x via set! → invalide :x puis :y, déclenche le callback sur :y
NeuroDSL.set!(g, :x, randn(Float32, 2,2))   # Message "y réagit au changement de :x" s'affiche

y réagit au changement de :x


JuliusGraph(Dict{Symbol, Dict{Symbol, GraphNode{Float32}}}(:default => Dict(:y => GraphNode{Float32}(:y, Float32[0.0 0.0; 0.0 0.0], nothing, false, false, Quantom, false, :default, Dict{Symbol, Any}(), Symbol[], var"#13#14"()), :x => GraphNode{Float32}(:x, Float32[1.0028478 -0.49982393; -0.619688 -0.16327277], nothing, false, false, Quantom, false, :default, Dict{Symbol, Any}(), [:y], nothing))), Dict{Symbol, Dict{Symbol, GraphRule}}(:default => Dict()), Dict{Symbol, Function}(), Dict{Symbol, Union{Nothing, Vector{Symbol}}}(:default => nothing), :default, Main.NeuroDSL.Backend.CUDADevice())

In [6]:
using .NeuroDSL, Random

function test_watch_early_stopping_converged()
    dev = NeuroDSL.Backend.CPUDevice()
    g = NeuroDSL.JuliusGraph(device=dev)

    Random.seed!(123)
    X = randn(Float32, 100, 4)
    Y = 2.0f0 .* X[:, 1:1] .+ 0.1f0 .* randn(Float32, 100, 1)

    NeuroDSL.set!(g, :x, X)
    NeuroDSL.set!(g, :y, Y; atom_type=NeuroDSL.Datom)

    out = NeuroDSL.Linear(4, 1, bias=false)(g, :x, :fc)
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [out, :y], :mse_loss))

    # Watcher : early stopping quand loss < 0.01
    stopped = Ref(false)
    NeuroDSL.node(g, loss_sym).on_change = (graph, sym, ns) -> begin
        val = graph.nodes[ns][sym].value
        if val !== nothing && val[1] < 0.01f0
            stopped[] = true
            println("⏹ Early stopping déclenché ! loss = ", val[1])
        end
    end

    ps = NeuroDSL.params(g)
    m1 = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    m2 = [NeuroDSL.Backend.zeros32(dev, size(p.value)...) for p in ps]
    lr = 0.5f0
    b1, b2, eps_v, wd, clip = 0.9f0, 0.999f0, 1f-8, 0.0f0, 1f0

    max_steps = 2000
    ctx = NeuroDSL.CtxStore()
    for t in 1:max_steps
        NeuroDSL.invalidate_all!(g)
        NeuroDSL.demand!(g, loss_sym; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx)

        # Appliquer l'optimiseur
        for (i, p) in enumerate(ps)
            NeuroDSL.adamw_step!(dev, p.value, p.gradient,
                                 m1[i], m2[i], lr, b1, b2, eps_v, t, clip, wd)
        end

        # Déclencher le callback via invalidation (la valeur est encore là)
        NeuroDSL._invalidate_downstream!(g, loss_sym, :default)

        if stopped[]
            println("Arrêt à l'étape $t (maximum $max_steps)")
            break
        end
    end
end

test_watch_early_stopping_converged()

⏹ Early stopping déclenché ! loss = 0.009516348
Arrêt à l'étape 56 (maximum 2000)


# Flux

In [7]:
using Flux, BenchmarkTools, Random

# Fonction de perte Flux (globale pour BenchmarkTools)
function loss_flux(model, x, y)
    return Flux.mse(model(x), y)
end

function compare_small_linear_cpu()
    in_dim, hidden_dim, out_dim = 512, 1024, 512
    batch_size = 64
    dev = NeuroDSL.Backend.CPUDevice()

    # ----- Flux (CPU) -----
    model = Chain(Dense(in_dim, hidden_dim, relu), Dense(hidden_dim, out_dim))
    x_f = randn(Float32, in_dim, batch_size)
    y_f = randn(Float32, out_dim, batch_size)

    # Warm‑up
    Flux.withgradient(m -> loss_flux(m, x_f, y_f), model)

    t_flux = @benchmark begin
        Flux.withgradient(m -> loss_flux(m, $x_f, $y_f), $model)
    end

    # ----- NeuroDSL (CPU) -----
    g = NeuroDSL.JuliusGraph(device=dev, namespace=:cmp_cpu)
    NeuroDSL.set!(g, :x, randn(Float32, batch_size, in_dim))
    NeuroDSL.set!(g, :y, zeros(Float32, batch_size, out_dim); atom_type=NeuroDSL.Datom)
    h1 = NeuroDSL.Linear(in_dim, hidden_dim)(g, :x, :fc1; namespace=:cmp_cpu)
    h2 = NeuroDSL.Linear(hidden_dim, out_dim)(g, h1, :fc2; namespace=:cmp_cpu)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [h2, :y], :mse_loss; namespace=:cmp_cpu))
    NeuroDSL._fuse!(g, [:h1, :h2]) 
    ctx = NeuroDSL.CtxStore()

    # Warm‑up
    NeuroDSL.demand!(g, :L; ctx_store=ctx, namespace=:cmp_cpu)
    NeuroDSL.backward_graph!(g, :L; ctx_store=ctx, namespace=:cmp_cpu, full=true)

    t_neuro = @benchmark begin
        NeuroDSL.invalidate_all!($g; namespace=:cmp_cpu)
        NeuroDSL.demand!($g, :L; ctx_store=$ctx, namespace=:cmp_cpu)
        NeuroDSL.backward_graph!($g, :L; ctx_store=$ctx, namespace=:cmp_cpu, full=true)
    end

    med_flux = median(t_flux).time * 1e-6
    med_neuro = median(t_neuro).time * 1e-6
    println("Flux CPU median : $(round(med_flux, digits=3)) ms")
    println("NeuroDSL CPU median : $(round(med_neuro, digits=3)) ms")
    println("Ratio (NeuroDSL / Flux) : $(round(med_neuro/med_flux, digits=2))x")
end

compare_small_linear_cpu()

Flux CPU median : 8.857 ms
NeuroDSL CPU median : 13.337 ms
Ratio (NeuroDSL / Flux) : 1.51x


In [8]:
using Flux, BenchmarkTools, CUDA, Random, Functors

# Fonction de perte (globale)
loss_flux(model, x, y) = Flux.mse(model(x), y)

function compare_small_linear_gpu_fair()
    in_dim, hidden_dim, out_dim = 512, 1024, 512
    batch_size = 64
    dev = NeuroDSL.Backend.CUDADevice()

    # ----- Flux GPU (transfert manuel robuste) -----
    model_cpu = Chain(Dense(in_dim, hidden_dim, relu), Dense(hidden_dim, out_dim))
    model = Functors.fmap(CUDA.cu, model_cpu)   # ← Correction ici
    x_f = CUDA.randn(Float32, in_dim, batch_size)
    y_f = CUDA.randn(Float32, out_dim, batch_size)

    @assert model.layers[1].weight isa CUDA.CuArray "Le modèle n'est pas sur GPU !"
    @assert x_f isa CUDA.CuArray "x_f n'est pas sur GPU !"

    # Warm‑up
    Flux.withgradient(m -> loss_flux(m, x_f, y_f), model)
    CUDA.synchronize()

    t_flux = @benchmark begin
        Flux.withgradient(m -> loss_flux(m, $x_f, $y_f), $model)
        CUDA.synchronize()
    end

    # ----- NeuroDSL GPU -----
    g = NeuroDSL.JuliusGraph(device=dev, namespace=:cmp_gpu)
    NeuroDSL.set!(g, :x, CUDA.randn(Float32, batch_size, in_dim))
    NeuroDSL.set!(g, :y, CUDA.zeros(Float32, batch_size, out_dim); atom_type=NeuroDSL.Datom)
    h1 = NeuroDSL.Linear(in_dim, hidden_dim)(g, :x, :fc1; namespace=:cmp_gpu)
    h2 = NeuroDSL.Linear(hidden_dim, out_dim)(g, h1, :fc2; namespace=:cmp_gpu)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:L, [h2, :y], :mse_loss; namespace=:cmp_gpu))
    ctx = NeuroDSL.CtxStore()

    NeuroDSL.demand!(g, :L; ctx_store=ctx, namespace=:cmp_gpu)
    NeuroDSL.backward_graph!(g, :L; ctx_store=ctx, namespace=:cmp_gpu, full=true)
    NeuroDSL._fuse!(g, [:h1, :h2]) 
    CUDA.synchronize()

    t_neuro = @benchmark begin
        NeuroDSL.invalidate_all!($g; namespace=:cmp_gpu)
        NeuroDSL.demand!($g, :L; ctx_store=$ctx, namespace=:cmp_gpu)
        NeuroDSL.backward_graph!($g, :L; ctx_store=$ctx, namespace=:cmp_gpu, full=true)
        CUDA.synchronize()
    end

    med_flux = median(t_flux).time * 1e-6
    med_neuro = median(t_neuro).time * 1e-6
    println("Flux GPU median : $(round(med_flux, digits=3)) ms")
    println("NeuroDSL GPU median : $(round(med_neuro, digits=3)) ms")
    println("Ratio (NeuroDSL / Flux) : $(round(med_neuro/med_flux, digits=2))x")
end

compare_small_linear_gpu_fair()

Flux GPU median : 1.44 ms
NeuroDSL GPU median : 2.712 ms
Ratio (NeuroDSL / Flux) : 1.88x


In [9]:
using Flux, BenchmarkTools, CUDA, Random, LinearAlgebra

# ----- Bloc Transformer simplifié pour Flux -----
struct SimpleTransformerBlock
    Wq::AbstractMatrix{Float32}
    Wk::AbstractMatrix{Float32}
    Wv::AbstractMatrix{Float32}
    Wo::AbstractMatrix{Float32}
    W1::AbstractMatrix{Float32}
    W2::AbstractMatrix{Float32}
    W3::AbstractMatrix{Float32}
    gamma1::AbstractVector{Float32}
    gamma2::AbstractVector{Float32}
end

function (b::SimpleTransformerBlock)(x::AbstractMatrix{Float32})  # x: (seq, dim)
    # RMSNorm
    x_norm1 = rmsnorm_flux(x, b.gamma1)
    # Attention mono‑tête
    q = x_norm1 * b.Wq'
    k = x_norm1 * b.Wk'
    v = x_norm1 * b.Wv'
    scale = 1f0 / sqrt(size(q,2))
    att = softmax(q * k' .* scale; dims=2)
    att_out = att * v * b.Wo'
    x = x + att_out
    # RMSNorm
    x_norm2 = rmsnorm_flux(x, b.gamma2)
    # MLP SwiGLU
    gate = x_norm2 * b.W1'
    up = x_norm2 * b.W3'
    swiglu = gate .* Flux.sigmoid_fast(gate) .* up
    mlp_out = swiglu * b.W2'
    return x + mlp_out
end

function rmsnorm_flux(x, gamma; eps=1f-6)
    nc = size(x,2)
    ms = sum(x.^2, dims=2) ./ Float32(nc) .+ eps
    rms_inv = @. 1f0 / sqrt(ms)
    return x .* rms_inv .* gamma'
end

# Initialisation aléatoire (sur CPU ou GPU)
function create_flux_block(dim, device=cpu)
    Wq = randn(Float32, dim, dim) .* 0.01f0 |> device
    Wk = randn(Float32, dim, dim) .* 0.01f0 |> device
    Wv = randn(Float32, dim, dim) .* 0.01f0 |> device
    Wo = randn(Float32, dim, dim) .* 0.01f0 |> device
    W1 = randn(Float32, dim, dim) .* 0.01f0 |> device
    W2 = randn(Float32, dim, dim) .* 0.01f0 |> device
    W3 = randn(Float32, dim, dim) .* 0.01f0 |> device
    gamma1 = ones(Float32, dim) |> device
    gamma2 = ones(Float32, dim) |> device
    return SimpleTransformerBlock(Wq,Wk,Wv,Wo,W1,W2,W3,gamma1,gamma2)
end

create_flux_block (generic function with 2 methods)

In [10]:
function build_transformer_block_neuro(g::NeuroDSL.JuliusGraph, x_sym::Symbol, dim::Int, ns::Symbol)
    # Paramètres
    NeuroDSL.set!(g, :Wq, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :Wk, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :Wv, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :Wo, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :W1, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :W2, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :W3, randn(Float32, dim, dim).*0.01f0; is_param=true, namespace=ns)
    NeuroDSL.set!(g, :gamma1, ones(Float32, dim); is_param=true, namespace=ns)
    NeuroDSL.set!(g, :gamma2, ones(Float32, dim); is_param=true, namespace=ns)

    # RMSNorm 1
    xn1 = Symbol(x_sym, :_norm1)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(xn1, [x_sym, :gamma1], :rmsnorm; namespace=ns))
    # Q, K, V
    q = Symbol(x_sym, :_q)
    k = Symbol(x_sym, :_k)
    v = Symbol(x_sym, :_v)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(q, [xn1, :Wq], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(k, [xn1, :Wk], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(v, [xn1, :Wv], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    # Scores
    scores = Symbol(x_sym, :_scores)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(scores, [q, k], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    # Scale mask
    sm = Symbol(x_sym, :_sm)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(sm, [scores], :scale_mask; attrs=Dict(:d_head=>dim), namespace=ns))
    # Softmax
    att = Symbol(x_sym, :_att)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(att, [sm], :softmax; namespace=ns))
    # Attention output
    ao = Symbol(x_sym, :_ao)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(ao, [att, v], :matmul; namespace=ns))
    # Output projection
    out_att = Symbol(x_sym, :_out_att)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out_att, [ao, :Wo], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    # Residual 1
    r1 = Symbol(x_sym, :_r1)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(r1, [x_sym, out_att], :add; namespace=ns))
    # RMSNorm 2
    xn2 = Symbol(x_sym, :_norm2)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(xn2, [r1, :gamma2], :rmsnorm; namespace=ns))
    # MLP gate & up
    gate = Symbol(x_sym, :_gate)
    up = Symbol(x_sym, :_up)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(gate, [xn2, :W1], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(up, [xn2, :W3], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    # SwiGLU
    sg = Symbol(x_sym, :_sg)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(sg, [gate, up], :swiglu; namespace=ns))
    # MLP output
    mlp_out = Symbol(x_sym, :_mlp_out)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(mlp_out, [sg, :W2], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    # Residual 2
    out = Symbol(x_sym, :_out)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out, [r1, mlp_out], :add; namespace=ns))
    return out
end

build_transformer_block_neuro (generic function with 1 method)

In [11]:
function bench_transformer_block(;dim=256, seq=64, dev=NeuroDSL.Backend.CPUDevice())
    ns = :bench_tb
    g = NeuroDSL.JuliusGraph(device=dev, namespace=ns)
    x_sym = :x
    NeuroDSL.set!(g, x_sym, NeuroDSL.Backend.randn32(dev, seq, dim))
    out_sym = build_transformer_block_neuro(g, x_sym, dim, ns)
    loss_sym = :loss
    NeuroDSL.set!(g, :y, NeuroDSL.Backend.zeros32(dev, seq, dim); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [out_sym, :y], :mse_loss; namespace=ns))

    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    if dev isa NeuroDSL.Backend.CUDADevice
        CUDA.synchronize()
    end

    t_neuro = @benchmark begin
        NeuroDSL.invalidate_all!($g; namespace=$ns)
        NeuroDSL.demand!($g, $loss_sym; ctx_store=$ctx, namespace=$ns)
        NeuroDSL.backward_graph!($g, $loss_sym; ctx_store=$ctx, namespace=$ns, full=true)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end

    # ---- Flux benchmark ----
    if dev isa NeuroDSL.Backend.CPUDevice
        flux_dev = identity
        flux_dev_name = "CPU"
    else
        flux_dev = CUDA.cu
        flux_dev_name = "GPU"
    end
    block_flux = create_flux_block(dim, flux_dev)
    x_flux = randn(Float32, seq, dim) |> flux_dev
    y_flux = zeros(Float32, seq, dim) |> flux_dev
    # Fonction de perte locale capturant y_flux
    loss_fn = (m, x) -> Flux.mse(m(x), y_flux)
    Flux.withgradient(m -> loss_fn(m, x_flux), block_flux)  # warmup
    if dev isa NeuroDSL.Backend.CUDADevice
        CUDA.synchronize()
    end

    t_flux = @benchmark begin
        Flux.withgradient(m -> $loss_fn(m, $x_flux), $block_flux)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end

    med_neuro = median(t_neuro).time * 1e-6
    med_flux = median(t_flux).time * 1e-6
    println("Device: $flux_dev_name")
    println("Flux median : $(round(med_flux, digits=3)) ms")
    println("NeuroDSL median : $(round(med_neuro, digits=3)) ms")
    println("Ratio (NeuroDSL / Flux) : $(round(med_neuro/med_flux, digits=2))x")
end

# Test CPU
bench_transformer_block(;dim=256, seq=64, dev=NeuroDSL.Backend.CPUDevice())
# Test GPU (décommente si CUDA est disponible)
# bench_transformer_block(;dim=256, seq=64, dev=NeuroDSL.Backend.CUDADevice())

Device: CPU
Flux median : 40.068 ms
NeuroDSL median : 40.019 ms
Ratio (NeuroDSL / Flux) : 1.0x


In [12]:
bench_transformer_block(;dim=256, seq=64, dev=NeuroDSL.Backend.CUDADevice())

Device: GPU
Flux median : 9.779 ms
NeuroDSL median : 5.926 ms
Ratio (NeuroDSL / Flux) : 0.61x


In [9]:
bench_transformer_block(;dim=512, seq=128, dev=NeuroDSL.Backend.CUDADevice())

Device: GPU
Flux median : 14.119 ms
NeuroDSL median : 6.333 ms
Ratio (NeuroDSL / Flux) : 0.45x


In [13]:
using Flux, BenchmarkTools, CUDA, Random, LinearAlgebra, Printf

# ── Utilitaires Flux (identiques au notebook) ──────────────────────────────

function rmsnorm_flux(x, gamma; eps=1f-6)
    nc = size(x, 2)
    ms = sum(x.^2, dims=2) ./ Float32(nc) .+ eps
    rms_inv = @. 1f0 / sqrt(ms)
    return x .* rms_inv .* gamma'
end

struct SimpleTransformerBlock
    Wq::AbstractMatrix{Float32}
    Wk::AbstractMatrix{Float32}
    Wv::AbstractMatrix{Float32}
    Wo::AbstractMatrix{Float32}
    W1::AbstractMatrix{Float32}
    W2::AbstractMatrix{Float32}
    W3::AbstractMatrix{Float32}
    gamma1::AbstractVector{Float32}
    gamma2::AbstractVector{Float32}
end

function (b::SimpleTransformerBlock)(x::AbstractMatrix{Float32})
    x_norm1 = rmsnorm_flux(x, b.gamma1)
    q = x_norm1 * b.Wq'
    k = x_norm1 * b.Wk'
    v = x_norm1 * b.Wv'
    scale = 1f0 / sqrt(Float32(size(q, 2)))
    att = softmax(q * k' .* scale; dims=2)
    att_out = att * v * b.Wo'
    x = x + att_out
    x_norm2 = rmsnorm_flux(x, b.gamma2)
    gate = x_norm2 * b.W1'
    up   = x_norm2 * b.W3'
    swiglu = gate .* Flux.sigmoid_fast(gate) .* up
    mlp_out = swiglu * b.W2'
    return x + mlp_out
end

function create_flux_block(dim, device=identity)
    init = () -> randn(Float32, dim, dim) .* 0.01f0 |> device
    SimpleTransformerBlock(
        init(), init(), init(), init(),
        init(), init(), init(),
        ones(Float32, dim) |> device,
        ones(Float32, dim) |> device
    )
end

# ── Construction du bloc NeuroDSL ──────────────────────────────────────────
# CORRECTION : namespace dynamique (Symbol unique par appel) pour éviter
# les collisions quand bench_transformer_block est appelé plusieurs fois.

function build_transformer_block_neuro(g, x_sym, dim, ns)
    for (sym, sz) in [
        (:Wq,(dim,dim)), (:Wk,(dim,dim)), (:Wv,(dim,dim)), (:Wo,(dim,dim)),
        (:W1,(dim,dim)), (:W2,(dim,dim)), (:W3,(dim,dim))
    ]
        NeuroDSL.set!(g, sym, randn(Float32, sz...) .* 0.01f0;
                      is_param=true, namespace=ns)
    end
    NeuroDSL.set!(g, :gamma1, ones(Float32, dim); is_param=true, namespace=ns)
    NeuroDSL.set!(g, :gamma2, ones(Float32, dim); is_param=true, namespace=ns)

    add!(op, ins, out) = NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(out, ins, op; namespace=ns))
    adda!(op, ins, out; kw...) = NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(out, ins, op; attrs=Dict(kw...), namespace=ns))

    add!(:rmsnorm, [x_sym, :gamma1], :xn1)
    adda!(:matmul,  [:xn1, :Wq], :q; trans_b=true)
    adda!(:matmul,  [:xn1, :Wk], :k; trans_b=true)
    adda!(:matmul,  [:xn1, :Wv], :v; trans_b=true)
    adda!(:matmul,  [:q,   :k],  :scores; trans_b=true)
    adda!(:scale_mask, [:scores], :sm; d_head=dim)
    add!(:softmax,  [:sm], :att)
    add!(:matmul,   [:att, :v], :ao)
    adda!(:matmul,  [:ao, :Wo], :out_att; trans_b=true)
    add!(:add,      [x_sym, :out_att], :r1)
    add!(:rmsnorm,  [:r1, :gamma2], :xn2)
    adda!(:matmul,  [:xn2, :W1], :gate; trans_b=true)
    adda!(:matmul,  [:xn2, :W3], :up;   trans_b=true)
    add!(:swiglu,   [:gate, :up], :sg)
    adda!(:matmul,  [:sg, :W2],  :mlp_out; trans_b=true)
    add!(:add,      [:r1, :mlp_out], :out)
    return :out
end

# ── Benchmark principal ────────────────────────────────────────────────────

function bench_transformer_block(;
        dim::Int,
        seq::Int = 64,
        dev = NeuroDSL.Backend.CPUDevice(),
        n_samples::Int = 50)          # BenchmarkTools samples

    # Namespace unique : évite tout conflit entre appels successifs
    ns = Symbol(:tb_, dim, :_, seq, :_,
                dev isa NeuroDSL.Backend.CUDADevice ? :gpu : :cpu)

    dev_name = dev isa NeuroDSL.Backend.CUDADevice ? "GPU" : "CPU"
    println("\n━━━  dim=$dim  seq=$seq  $dev_name  ━━━")

    # ── NeuroDSL ──────────────────────────────────────────────────────────
    g = NeuroDSL.JuliusGraph(device=dev, namespace=ns)
    NeuroDSL.set!(g, :x, NeuroDSL.Backend.randn32(dev, seq, dim))

    out_sym  = build_transformer_block_neuro(g, :x, dim, ns)
    loss_sym = :loss

    NeuroDSL.set!(g, :y,
                  NeuroDSL.Backend.zeros32(dev, seq, dim);
                  atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(loss_sym, [out_sym, :y], :mse_loss; namespace=ns))

    ctx = NeuroDSL.CtxStore()
    # warmup (2 passes)
    for _ in 1:2
        NeuroDSL.invalidate_all!(g; namespace=ns)
        NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
        NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    end
    dev isa NeuroDSL.Backend.CUDADevice && CUDA.synchronize()

    t_neuro = @benchmark begin
        NeuroDSL.invalidate_all!($g; namespace=$ns)
        NeuroDSL.demand!($g, $loss_sym; ctx_store=$ctx, namespace=$ns)
        NeuroDSL.backward_graph!($g, $loss_sym; ctx_store=$ctx, namespace=$ns, full=true)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end samples=n_samples evals=1

    # ── Flux ──────────────────────────────────────────────────────────────
    flux_dev    = dev isa NeuroDSL.Backend.CUDADevice ? CUDA.cu : identity
    block_flux  = create_flux_block(dim, flux_dev)
    x_flux      = randn(Float32, seq, dim) |> flux_dev
    y_flux      = zeros(Float32, seq, dim) |> flux_dev
    loss_fn     = (m, x) -> Flux.mse(m(x), y_flux)

    # warmup
    Flux.withgradient(m -> loss_fn(m, x_flux), block_flux)
    dev isa NeuroDSL.Backend.CUDADevice && CUDA.synchronize()

    t_flux = @benchmark begin
        Flux.withgradient(m -> $loss_fn(m, $x_flux), $block_flux)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end samples=n_samples evals=1

    med_n = median(t_neuro).time * 1e-6   # ns → ms
    med_f = median(t_flux).time  * 1e-6
    ratio = med_n / med_f

    @printf "  Flux     médiane : %8.3f ms\n"  med_f
    @printf "  NeuroDSL médiane : %8.3f ms\n"  med_n
    @printf "  Ratio (N/F)      :     %.2fx  %s\n" ratio (ratio < 1 ? "✅ NeuroDSL PLUS RAPIDE" : ratio < 1.15 ? "≈ parité" : "⚠ NeuroDSL plus lent")

    return (dim=dim, seq=seq, device=dev_name,
            flux_ms=med_f, neuro_ms=med_n, ratio=ratio)
end

# ── Série complète ─────────────────────────────────────────────────────────

println("="^55)
println("  Benchmark Transformer Block — NeuroDSL vs Flux")
println("="^55)

results = []

# GPU — les trois dimensions
# Note : on évite les noms `gpu` et `cpu` — Flux les exporte dans Main,
# toute assignation globale avec ces noms lève une erreur Julia.
gpu_dev = NeuroDSL.Backend.CUDADevice()
for dim in [256, 512, 1024]
    push!(results, bench_transformer_block(dim=dim, seq=64, dev=gpu_dev))
end

# CPU — seulement 256 et 512 (1024 CPU est trop lent pour un benchmark propre)
cpu_dev = NeuroDSL.Backend.CPUDevice()
for dim in [256, 512]
    push!(results, bench_transformer_block(dim=dim, seq=64, dev=cpu_dev, n_samples=20))
end

# ── Tableau récapitulatif ──────────────────────────────────────────────────
println("\n")
println("┌─────────────────────────────────────────────────────────┐")
println("│          Récapitulatif — médiane forward+backward        │")
println("├──────┬──────┬──────┬──────────┬──────────┬─────────────┤")
println("│  dim │  seq │  dev │  Flux ms │ NeuroDSL │    ratio    │")
println("├──────┼──────┼──────┼──────────┼──────────┼─────────────┤")
for r in results
    verdict = r.ratio < 1.0 ? "✅ +$(round((1-r.ratio)*100, digits=1))%" :
              r.ratio < 1.15 ? "≈ parité" :
              "⚠ ×$(round(r.ratio, digits=2))"
    @printf "│ %4d │  %3d │ %3s  │ %8.3f │ %8.3f │ %-11s │\n"  r.dim r.seq r.device r.flux_ms r.neuro_ms verdict
end
println("└──────┴──────┴──────┴──────────┴──────────┴─────────────┘")
println("  ✅ = NeuroDSL plus rapide  |  ⚠ = NeuroDSL plus lent")

  Benchmark Transformer Block — NeuroDSL vs Flux

━━━  dim=256  seq=64  GPU  ━━━
  Flux     médiane :   11.577 ms
  NeuroDSL médiane :    6.243 ms
  Ratio (N/F)      :     0.54x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=512  seq=64  GPU  ━━━
  Flux     médiane :    8.209 ms
  NeuroDSL médiane :    6.335 ms
  Ratio (N/F)      :     0.77x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=1024  seq=64  GPU  ━━━
  Flux     médiane :    8.814 ms
  NeuroDSL médiane :    6.491 ms
  Ratio (N/F)      :     0.74x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=256  seq=64  CPU  ━━━
  Flux     médiane :   38.131 ms
  NeuroDSL médiane :   42.068 ms
  Ratio (N/F)      :     1.10x  ≈ parité

━━━  dim=512  seq=64  CPU  ━━━
  Flux     médiane :   45.181 ms
  NeuroDSL médiane :   53.744 ms
  Ratio (N/F)      :     1.19x  ⚠ NeuroDSL plus lent


┌─────────────────────────────────────────────────────────┐
│          Récapitulatif — médiane forward+backward        │
├──────┬──────┬──────┬──────────┬──────────┬─────────────┤
│  dim │  seq │  de

# NeuroDSL vs Pytorch 

In [18]:
function Base.similar(a::CUDA.CuArray{Float32})
    arr = CUDA.zeros(Float32, size(a)...)
    MemTrack.alloc!(sizeof(arr))
    finalizer(x -> MemTrack.free!(sizeof(x)), arr)
    return arr
end

# Vérifie
peak = peak_mem() do
    x = NeuroDSL.Backend.zeros32(NeuroDSL.Backend.CUDADevice(), 1000, 1000)
    y = similar(x)  # appelle similar(a::CuArray{Float32}) sans type
end
@show peak  # doit afficher ~7.63

peak = 7.62939453125


7.62939453125

In [17]:
function theoretical_peak_memory(depth, dim, batch, every)
    ns = :theo
    g = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice(), namespace=ns)
    loss_sym = build_deep_network(g, depth, dim, batch; ns=ns)
    
    # --- Baseline (sans checkpointing) ---
    plan_base, _ = NeuroDSL.plan_memory!(g; namespace=ns)
    n_slots_base = plan_base.n_slots
    tensor_bytes = batch * dim * sizeof(Float32)
    peak_base_MiB = n_slots_base * tensor_bytes / 1024^2
    
    # --- Checkpointing ---
    cd = NeuroDSL.CheckpointData(every=every)
    sch = NeuroDSL.CheckpointSchedule(g, cd; namespace=ns)
    # Nombre de checkpoints (activations conservées)
    n_checkpoints = length(sch.checkpoints)
    # Longueur maximale d'un segment à recomputer
    order = sch.order
    checkpoint_positions = [i for (i, sym) in enumerate(order) if sym in sch.checkpoints]
    if isempty(checkpoint_positions)
        max_segment = length(order)
    else
        positions = [0; checkpoint_positions; length(order)+1]
        max_segment = maximum(diff(positions))
    end
    # Pic = checkpoints + segment le plus long (moins 1 car le dernier checkpoint est inclus)
    n_slots_ckpt = n_checkpoints + max_segment - 1
    peak_ckpt_MiB = n_slots_ckpt * tensor_bytes / 1024^2
    
    return (peak_base_MiB, peak_ckpt_MiB, (peak_base_MiB - peak_ckpt_MiB) / peak_base_MiB * 100)
end

# Exemple avec depth=20, dim=1024, batch=64, every=4
depth, dim, batch, every = 20, 1024, 64, 4
base, ckpt, eco = theoretical_peak_memory(depth, dim, batch, every)
@printf "Baseline (sans checkpointing) : %.2f MiB\n" base
@printf "Checkpointed (every=%d)        : %.2f MiB\n" every ckpt
@printf "Économie mémoire              : %.1f%%\n" eco

Baseline (sans checkpointing) : 15.75 MiB


[ Info: CheckpointSchedule: 33 checkpoints, 30 recomputables


Checkpointed (every=4)        : 9.00 MiB
Économie mémoire              : 42.9%


In [18]:
# ── Memory Tracker — module dédié ────────────────────────────
module MemTrack
    const peak    = Ref{Int64}(0)
    const current = Ref{Int64}(0)
    const active  = Ref{Bool}(false)

    reset!() = (peak[] = 0; current[] = 0; active[] = true)
    stop!()  = (active[] = false)

    function alloc!(nbytes::Int)
        active[] || return
        current[] += nbytes
        current[] > peak[] && (peak[] = current[])
    end
    peak_mb() = peak[] / 1024^2
end

# ── Fonction de mesure ────────────────────────────────────────
function peak_mem(f::Function)
    MemTrack.reset!()
    f()
    CUDA.synchronize()
    MemTrack.stop!()
    return MemTrack.peak_mb()
end

peak_mem (generic function with 1 method)

In [16]:
using CUDA, Printf

# ═══════════════════════════════════════════════════════════
# MEMORY TRACKER — compte les octets alloués par NeuroDSL
# ═══════════════════════════════════════════════════════════
module MemTrack
    const peak    = Ref{Int64}(0)
    const current = Ref{Int64}(0)
    const active  = Ref{Bool}(false)

    reset!() = (peak[] = 0; current[] = 0; active[] = true)
    stop!()  = (active[] = false)

    function alloc!(nbytes::Int)
        active[] || return
        current[] += nbytes
        current[] > peak[] && (peak[] = current[])
    end
    function free!(nbytes::Int)
        active[] || return
        current[] = max(0, current[] - nbytes)
    end
    peak_mb()    = peak[]    / 1024^2
    current_mb() = current[] / 1024^2
end

# ── Patch Backend (intercept toutes les allocations GPU) ───
function NeuroDSL.Backend.zeros32(d::NeuroDSL.Backend.CUDADevice, dims...)
    a = CUDA.zeros(Float32, dims...)
    MemTrack.alloc!(sizeof(a))
    finalizer(x -> MemTrack.free!(sizeof(x)), a)
    return a
end
function NeuroDSL.Backend.ones32(d::NeuroDSL.Backend.CUDADevice, dims...)
    a = CUDA.ones(Float32, dims...)
    MemTrack.alloc!(sizeof(a))
    finalizer(x -> MemTrack.free!(sizeof(x)), a)
    return a
end
function NeuroDSL.Backend.rand32(d::NeuroDSL.Backend.CUDADevice, dims...)
    a = CUDA.rand(Float32, dims...)
    MemTrack.alloc!(sizeof(a))
    finalizer(x -> MemTrack.free!(sizeof(x)), a)
    return a
end
function NeuroDSL.Backend.randn32(d::NeuroDSL.Backend.CUDADevice, dims...)
    a = CUDA.randn(Float32, dims...)
    MemTrack.alloc!(sizeof(a))
    finalizer(x -> MemTrack.free!(sizeof(x)), a)
    return a
end

# Patch similar pour CuArray (utilisé dans accum_grad! et buffers internes)
function Base.similar(a::CUDA.CuArray, ::Type{T},
                      dims::Tuple{Union{Integer,Base.OneTo},
                                  Vararg{Union{Integer,Base.OneTo}}}) where T
    arr = CUDA.zeros(T, map(d -> d isa Base.OneTo ? length(d) : d, dims)...)
    if T == Float32
        MemTrack.alloc!(sizeof(arr))
        finalizer(x -> MemTrack.free!(sizeof(x)), arr)
    end
    return arr
end
function Base.similar(a::CUDA.CuArray{Float32})
    arr = CUDA.zeros(Float32, size(a)...)
    MemTrack.alloc!(sizeof(arr))
    finalizer(x -> MemTrack.free!(sizeof(x)), arr)
    return arr
end

# ── Fonction de mesure ─────────────────────────────────────
function peak_mem(f::Function)
    GC.gc(); CUDA.synchronize()
    MemTrack.reset!()
    f()
    CUDA.synchronize(); GC.gc()
    MemTrack.stop!()
    return MemTrack.peak_mb()
end

# ═══════════════════════════════════════════════════════════
# FIX _recompute_segment! — ne recompute jamais params/sources
# ═══════════════════════════════════════════════════════════
function NeuroDSL._recompute_segment!(g::NeuroDSL.JuliusGraph, target_sym::Symbol,
                                       schedule::NeuroDSL.CheckpointSchedule,
                                       ctx_store=nothing;
                                       namespace::Symbol=g.active_ns)
    ns = namespace
    nd = get(g.nodes[ns], target_sym, nothing)
    nd === nothing && return nothing

    # Params et sources ne se recomputent pas — ils sont toujours en mémoire
    if nd.is_param || !haskey(g.rules[ns], target_sym)
        nd.valid = true
        return nd.value
    end

    order = schedule.order
    target_idx = findfirst(==(target_sym), order)
    target_idx === nothing && return nothing

    # Cherche le dernier nœud valide avant la cible
    start_idx = 1
    for i in (target_idx-1):-1:1
        prev_nd = get(g.nodes[ns], order[i], nothing)
        if prev_nd !== nothing && prev_nd.value !== nothing && prev_nd.valid
            start_idx = i + 1
            break
        end
    end

    for i in start_idx:target_idx
        sym  = order[i]
        nd_i = get(g.nodes[ns], sym, nothing)
        nd_i === nothing && continue
        nd_i.valid && nd_i.value !== nothing && continue
        haskey(g.rules[ns], sym) || continue
        NeuroDSL.demand!(g, sym; ctx_store=ctx_store, namespace=ns)
    end
    return g.nodes[ns][target_sym].value
end

# ═══════════════════════════════════════════════════════════
# FIX backward_with_checkpointing! — libère les recomputables
# après usage avec use_count exact
# ═══════════════════════════════════════════════════════════
function NeuroDSL.backward_with_checkpointing!(g::NeuroDSL.JuliusGraph, loss_sym::Symbol;
                                                ctx_store::NeuroDSL.CtxStore=NeuroDSL.CtxStore(),
                                                schedule::NeuroDSL.CheckpointSchedule,
                                                namespace::Symbol=g.active_ns)
    ns = namespace
    NeuroDSL.zero_grads!(g; namespace=ns)
    ln = NeuroDSL.node(g, loss_sym; namespace=ns)
    @assert length(ln.value) == 1 "loss doit être scalaire"
    ln.gradient = NeuroDSL.Backend.ones32(g.device, size(ln.value)...)

    # Précalcule le nombre de fois que chaque recomputable est utilisé comme input
    # dans les règles backward (= règles dont le nœud de sortie recevra un gradient)
    use_count = Dict{Symbol,Int}()
    for (rsym, rule2) in g.rules[ns]
        for s in rule2.inputs
            s ∈ schedule.recomputable || continue
            use_count[s] = get(use_count, s, 0) + 1
        end
    end
    remaining = copy(use_count)

    for out_sym in reverse(schedule.order)
        !haskey(g.rules[ns], out_sym) && continue
        rule   = g.rules[ns][out_sym]
        nd_out = g.nodes[ns][out_sym]
        nd_out.gradient === nothing && continue
        !haskey(NeuroDSL.GRAD_RULES, rule.op) &&
            error("❌ Pas de règle backward pour :$(rule.op)")

        # Recompute les entrées manquantes (seulement les recomputables vrais)
        for in_sym in rule.inputs
            in_nd = get(g.nodes[ns], in_sym, nothing)
            in_nd === nothing && continue
            in_nd.is_param && continue                          # params toujours en mémoire
            haskey(g.rules[ns], in_sym) || continue            # sources toujours en mémoire
            if in_nd.value === nothing || !in_nd.valid
                NeuroDSL._recompute_segment!(g, in_sym, schedule, ctx_store; namespace=ns)
            end
        end

        ctx = get(ctx_store, out_sym, nothing)
        if ctx === nothing
            ctx_tmp = NeuroDSL.CtxStore()
            NeuroDSL.execute_rule!(g, rule; ctx_store=ctx_tmp)
            ctx = get(ctx_tmp, out_sym, Dict{Symbol,Any}())
        end

        inputs_vals = [g.nodes[ns][s].value for s in rule.inputs]
        grads = NeuroDSL.GRAD_RULES[rule.op](g.device, nd_out.gradient, ctx, inputs_vals)

        for (i, in_sym) in enumerate(rule.inputs)
            NeuroDSL.accum_grad!(g.nodes[ns][in_sym], grads[i])
        end

        # Décrémente use_count et libère les recomputables épuisés
        for in_sym in rule.inputs
            in_sym ∉ schedule.recomputable && continue
            remaining[in_sym] = get(remaining, in_sym, 1) - 1
            if remaining[in_sym] <= 0
                in_nd = g.nodes[ns][in_sym]
                if in_nd.value !== nothing
                    nb = sizeof(in_nd.value)
                    in_nd.value = nothing
                    in_nd.valid = false
                    MemTrack.free!(nb)
                end
            end
        end

        # Libère le gradient du nœud traité
        if nd_out.gradient !== nothing
            MemTrack.free!(sizeof(nd_out.gradient))
            nd_out.gradient = nothing
        end
        delete!(ctx_store, out_sym)
    end
    return g
end

# ═══════════════════════════════════════════════════════════
# MODÈLES
# ═══════════════════════════════════════════════════════════
function build_linear_model(g, M, D; ns=g.active_ns)
    NeuroDSL.set!(g, :X, CUDA.randn(Float32, M, D); namespace=ns)
    NeuroDSL.set!(g, :Y, CUDA.zeros(Float32, M, D); atom_type=NeuroDSL.Datom, namespace=ns)
    h1 = NeuroDSL.Linear(D, D, bias=false)(g, :X, :fc1; namespace=ns)
    h2 = NeuroDSL.LayerNorm(D)(g, h1, :norm; namespace=ns)
    h3 = NeuroDSL.Linear(D, D, bias=false)(g, h2, :fc2; namespace=ns)
    loss_sym = :loss
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [h3, :Y], :mse_loss; namespace=ns))
    return loss_sym
end

function build_transformer_block_neuro(g, x_sym, dim, ns)
    for (sym, sz) in [
        (:Wq,(dim,dim)), (:Wk,(dim,dim)), (:Wv,(dim,dim)), (:Wo,(dim,dim)),
        (:W1,(dim,dim)), (:W2,(dim,dim)), (:W3,(dim,dim))
    ]
        NeuroDSL.set!(g, sym, CUDA.randn(Float32, sz...) .* 0.01f0;
                      is_param=true, namespace=ns)
    end
    NeuroDSL.set!(g, :gamma1, CUDA.ones(Float32, dim); is_param=true, namespace=ns)
    NeuroDSL.set!(g, :gamma2, CUDA.ones(Float32, dim); is_param=true, namespace=ns)

    add!(op, ins, out)      = NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out, ins, op; namespace=ns))
    adda!(op, ins, out; kw...) = NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out, ins, op; attrs=Dict(kw...), namespace=ns))

    add!(:rmsnorm,    [x_sym, :gamma1], :xn1)
    adda!(:matmul,    [:xn1, :Wq], :q;      trans_b=true)
    adda!(:matmul,    [:xn1, :Wk], :k;      trans_b=true)
    adda!(:matmul,    [:xn1, :Wv], :v;      trans_b=true)
    adda!(:matmul,    [:q,   :k],  :scores; trans_b=true)
    adda!(:scale_mask,[:scores],   :sm;     d_head=dim)
    add!(:softmax,    [:sm],       :att)
    add!(:matmul,     [:att, :v],  :ao)
    adda!(:matmul,    [:ao, :Wo],  :out_att; trans_b=true)
    add!(:add,        [x_sym, :out_att], :r1)
    add!(:rmsnorm,    [:r1, :gamma2],    :xn2)
    adda!(:matmul,    [:xn2, :W1], :gate; trans_b=true)
    adda!(:matmul,    [:xn2, :W3], :up;   trans_b=true)
    add!(:swiglu,     [:gate, :up], :sg)
    adda!(:matmul,    [:sg, :W2],  :mlp_out; trans_b=true)
    add!(:add,        [:r1, :mlp_out], :out)
    return :out
end

function build_deep_network(g, depth, dim, batch; ns=g.active_ns)
    NeuroDSL.set!(g, :x, CUDA.randn(Float32, batch, dim); namespace=ns)
    prev = :x
    for i in 1:depth
        w = Symbol(:W, i)
        NeuroDSL.set!(g, w, CUDA.randn(Float32, dim, dim) .* 0.01f0;
                      is_param=true, namespace=ns)
        mul = Symbol(:mul, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(mul, [prev, w], :matmul;
                          attrs=Dict(:trans_b=>true), namespace=ns))
        act = Symbol(:act, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(act, [mul], :relu; namespace=ns))
        prev = act
    end
    loss_sym = :loss
    NeuroDSL.set!(g, :target, CUDA.zeros(Float32, batch, dim);
                  atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [prev, :target], :mse_loss; namespace=ns))
    return loss_sym
end

# ── Helper mesure ──────────────────────────────────────────
function peak_mem_neurodsl(g, loss_sym, ctx, ns; use_checkpointing=false, every=3)
    if use_checkpointing
        cd  = NeuroDSL.CheckpointData(every=every)
        sch = NeuroDSL.CheckpointSchedule(g, cd; namespace=ns)
        return peak_mem() do
            NeuroDSL.invalidate_all!(g; namespace=ns)
            NeuroDSL.zero_grads!(g; namespace=ns)
            NeuroDSL.forward_with_checkpointing!(g, loss_sym, ctx, sch; namespace=ns)
            NeuroDSL.backward_with_checkpointing!(g, loss_sym;
                ctx_store=ctx, schedule=sch, namespace=ns)
            CUDA.synchronize()
        end
    else
        return peak_mem() do
            NeuroDSL.invalidate_all!(g; namespace=ns)
            NeuroDSL.zero_grads!(g; namespace=ns)
            NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
            NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
            CUDA.synchronize()
        end
    end
end

# ═══════════════════════════════════════════════════════════
# BENCHMARK 1 — Modèle linéaire
# ═══════════════════════════════════════════════════════════
println("="^62)
println("   CUDA — pic mémoire GPU — Linear→LayerNorm→Linear")
println("="^62)
println(rpad("config (M×D)", 20), rpad("pic ND (MB)", 15), rpad("pic PT (MB)", 15))
println("-"^62)

configs_linear = [(8,64), (32,128), (128,256), (512,512), (1024,512)]
pytorch_linear = Dict(
    (8,64) => 8.16, (32,128) => 0.16, (128,256) => 0.75,
    (512,512) => 4.00, (1024,512) => 6.00
)

for (M, D) in configs_linear
    ns = Symbol(:lin_, M, :_, D)
    g  = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice(), namespace=ns)
    loss_sym = build_linear_model(g, M, D; ns=ns)
    ctx = NeuroDSL.CtxStore()
    # Warm-up hors mesure
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    CUDA.synchronize()

    peak = peak_mem_neurodsl(g, loss_sym, ctx, ns)
    pt   = pytorch_linear[(M, D)]
    @printf "%-20s %12.2f %12.2f\n" "$(M)×$(D)" peak pt
    g = nothing; GC.gc(); CUDA.reclaim()
end

# ═══════════════════════════════════════════════════════════
# BENCHMARK 2 — Transformer block
# ═══════════════════════════════════════════════════════════
println()
println("="^62)
println("   CUDA — pic mémoire GPU — Transformer block (seq=64)")
println("="^62)
println(rpad("config", 20), rpad("pic ND (MB)", 15), rpad("pic PT (MB)", 15))
println("-"^62)

dims       = [256, 512, 1024]
seq        = 64
pytorch_tb = Dict(256 => 20.32, 512 => 31.38, 1024 => 74.51)

for dim in dims
    ns = Symbol(:tb_, dim)
    g  = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice(), namespace=ns)
    NeuroDSL.set!(g, :x, CUDA.randn(Float32, seq, dim); namespace=ns)
    out_sym  = build_transformer_block_neuro(g, :x, dim, ns)
    loss_sym = :loss
    NeuroDSL.set!(g, :y, CUDA.zeros(Float32, seq, dim); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(loss_sym, [out_sym, :y], :mse_loss; namespace=ns))
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    CUDA.synchronize()

    peak = peak_mem_neurodsl(g, loss_sym, ctx, ns)
    pt   = pytorch_tb[dim]
    @printf "  dim=%-15d %12.2f %12.2f\n" dim peak pt
    g = nothing; GC.gc(); CUDA.reclaim()
end

# ═══════════════════════════════════════════════════════════
# BENCHMARK 3 — Gradient checkpointing
# ═══════════════════════════════════════════════════════════
println()
println("="^62)
println("   Gradient checkpointing — pic mémoire (NeuroDSL)")
println("="^62)

ckpt_configs = [(10, 1024, 3, 64), (20, 2048, 4, 64)]

for (depth, dim, every, batch) in ckpt_configs
    ns = Symbol(:deep_, depth, :_, dim)
    g  = NeuroDSL.JuliusGraph(device=NeuroDSL.Backend.CUDADevice(), namespace=ns)
    loss_sym = build_deep_network(g, depth, dim, batch; ns=ns)
    ctx = NeuroDSL.CtxStore()
    NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
    NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    CUDA.synchronize()

    pic_full = peak_mem_neurodsl(g, loss_sym, ctx, ns; use_checkpointing=false)
    pic_ckpt = peak_mem_neurodsl(g, loss_sym, ctx, ns; use_checkpointing=true, every=every)

    eco = pic_full > 0 ? (pic_full - pic_ckpt) / pic_full * 100 : NaN
    @printf "\n  depth=%d  dim=%d  every=%d\n" depth dim every
    @printf "    Sans checkpointing : %.2f MB\n" pic_full
    @printf "    Avec checkpointing : %.2f MB\n" pic_ckpt
    @printf "    Économie           : %.2f MB (%.1f%%)\n" (pic_full - pic_ckpt) eco
    g = nothing; GC.gc(); CUDA.reclaim()
end

   CUDA — pic mémoire GPU — Linear→LayerNorm→Linear
config (M×D)        pic ND (MB)    pic PT (MB)    
--------------------------------------------------------------
8×64                         0.05         8.16
32×128                       0.24         0.16
128×256                      1.38         0.75
512×512                      9.00         4.00
1024×512                    16.00         6.00

   CUDA — pic mémoire GPU — Transformer block (seq=64)
config              pic ND (MB)    pic PT (MB)    
--------------------------------------------------------------
  dim=256                     3.47        20.32
  dim=512                    10.35        31.38
  dim=1024                   34.61        74.51

   Gradient checkpointing — pic mémoire (NeuroDSL)


[ Info: CheckpointSchedule: 20 checkpoints, 13 recomputables



  depth=10  dim=1024  every=3
    Sans checkpointing : 47.75 MB
    Avec checkpointing : 8.00 MB
    Économie           : 39.75 MB (83.2%)


[ Info: CheckpointSchedule: 33 checkpoints, 30 recomputables



  depth=20  dim=2048  every=4
    Sans checkpointing : 350.50 MB
    Avec checkpointing : 30.50 MB
    Économie           : 320.00 MB (91.3%)


# Fuse

In [4]:
using Flux, BenchmarkTools, CUDA, Random, LinearAlgebra, Printf

# ── Utilitaires Flux (identiques au notebook) ──────────────────────────────

function rmsnorm_flux(x, gamma; eps=1f-6)
    nc = size(x, 2)
    ms = sum(x.^2, dims=2) ./ Float32(nc) .+ eps
    rms_inv = @. 1f0 / sqrt(ms)
    return x .* rms_inv .* gamma'
end

struct SimpleTransformerBlock
    Wq::AbstractMatrix{Float32}
    Wk::AbstractMatrix{Float32}
    Wv::AbstractMatrix{Float32}
    Wo::AbstractMatrix{Float32}
    W1::AbstractMatrix{Float32}
    W2::AbstractMatrix{Float32}
    W3::AbstractMatrix{Float32}
    gamma1::AbstractVector{Float32}
    gamma2::AbstractVector{Float32}
end

function (b::SimpleTransformerBlock)(x::AbstractMatrix{Float32})
    x_norm1 = rmsnorm_flux(x, b.gamma1)
    q = x_norm1 * b.Wq'
    k = x_norm1 * b.Wk'
    v = x_norm1 * b.Wv'
    scale = 1f0 / sqrt(Float32(size(q, 2)))
    att = softmax(q * k' .* scale; dims=2)
    att_out = att * v * b.Wo'
    x = x + att_out
    x_norm2 = rmsnorm_flux(x, b.gamma2)
    gate = x_norm2 * b.W1'
    up   = x_norm2 * b.W3'
    swiglu = gate .* Flux.sigmoid_fast(gate) .* up
    mlp_out = swiglu * b.W2'
    return x + mlp_out
end

function create_flux_block(dim, device=identity)
    init = () -> randn(Float32, dim, dim) .* 0.01f0 |> device
    SimpleTransformerBlock(
        init(), init(), init(), init(),
        init(), init(), init(),
        ones(Float32, dim) |> device,
        ones(Float32, dim) |> device
    )
end

# ── Construction du bloc NeuroDSL ──────────────────────────────────────────
# CORRECTION : namespace dynamique (Symbol unique par appel) pour éviter
# les collisions quand bench_transformer_block est appelé plusieurs fois.

function build_transformer_block_neuro(g, x_sym, dim, ns)
    for (sym, sz) in [
        (:Wq,(dim,dim)), (:Wk,(dim,dim)), (:Wv,(dim,dim)), (:Wo,(dim,dim)),
        (:W1,(dim,dim)), (:W2,(dim,dim)), (:W3,(dim,dim))
    ]
        NeuroDSL.set!(g, sym, randn(Float32, sz...) .* 0.01f0;
                      is_param=true, namespace=ns)
    end
    NeuroDSL.set!(g, :gamma1, ones(Float32, dim); is_param=true, namespace=ns)
    NeuroDSL.set!(g, :gamma2, ones(Float32, dim); is_param=true, namespace=ns)

    add!(op, ins, out) = NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(out, ins, op; namespace=ns))
    adda!(op, ins, out; kw...) = NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(out, ins, op; attrs=Dict(kw...), namespace=ns))

    add!(:rmsnorm, [x_sym, :gamma1], :xn1)
    adda!(:matmul,  [:xn1, :Wq], :q; trans_b=true)
    adda!(:matmul,  [:xn1, :Wk], :k; trans_b=true)
    adda!(:matmul,  [:xn1, :Wv], :v; trans_b=true)
    adda!(:matmul,  [:q,   :k],  :scores; trans_b=true)
    adda!(:scale_mask, [:scores], :sm; d_head=dim)
    add!(:softmax,  [:sm], :att)
    add!(:matmul,   [:att, :v], :ao)
    adda!(:matmul,  [:ao, :Wo], :out_att; trans_b=true)
    add!(:add,      [x_sym, :out_att], :r1)
    add!(:rmsnorm,  [:r1, :gamma2], :xn2)
    adda!(:matmul,  [:xn2, :W1], :gate; trans_b=true)
    adda!(:matmul,  [:xn2, :W3], :up;   trans_b=true)
    add!(:swiglu,   [:gate, :up], :sg)
    adda!(:matmul,  [:sg, :W2],  :mlp_out; trans_b=true)
    add!(:add,      [:r1, :mlp_out], :out)
    return :out
end

# ── Benchmark principal ────────────────────────────────────────────────────

function bench_transformer_block(;
        dim::Int,
        seq::Int = 64,
        dev = NeuroDSL.Backend.CPUDevice(),
        n_samples::Int = 50)          # BenchmarkTools samples

    # Namespace unique : évite tout conflit entre appels successifs
    ns = Symbol(:tb_, dim, :_, seq, :_,
                dev isa NeuroDSL.Backend.CUDADevice ? :gpu : :cpu)

    dev_name = dev isa NeuroDSL.Backend.CUDADevice ? "GPU" : "CPU"
    println("\n━━━  dim=$dim  seq=$seq  $dev_name  ━━━")

    # ── NeuroDSL ──────────────────────────────────────────────────────────
    g = NeuroDSL.JuliusGraph(device=dev, namespace=ns)
    NeuroDSL.set!(g, :x, NeuroDSL.Backend.randn32(dev, seq, dim))

    out_sym  = build_transformer_block_neuro(g, :x, dim, ns)
    loss_sym = :loss

    NeuroDSL.set!(g, :y,
                  NeuroDSL.Backend.zeros32(dev, seq, dim);
                  atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(loss_sym, [out_sym, :y], :mse_loss; namespace=ns))

    ctx = NeuroDSL.CtxStore()
    # warmup (2 passes)
    for _ in 1:2
        NeuroDSL.invalidate_all!(g; namespace=ns)
        NeuroDSL.demand!(g, loss_sym; ctx_store=ctx, namespace=ns)
        NeuroDSL.backward_graph!(g, loss_sym; ctx_store=ctx, namespace=ns, full=true)
    end
    dev isa NeuroDSL.Backend.CUDADevice && CUDA.synchronize()

    t_neuro = @benchmark begin
        NeuroDSL.invalidate_all!($g; namespace=$ns)
        NeuroDSL.demand!($g, $loss_sym; ctx_store=$ctx, namespace=$ns)
        NeuroDSL.backward_graph!($g, $loss_sym; ctx_store=$ctx, namespace=$ns, full=true)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end samples=n_samples evals=1

    # ── Flux ──────────────────────────────────────────────────────────────
    flux_dev    = dev isa NeuroDSL.Backend.CUDADevice ? CUDA.cu : identity
    block_flux  = create_flux_block(dim, flux_dev)
    x_flux      = randn(Float32, seq, dim) |> flux_dev
    y_flux      = zeros(Float32, seq, dim) |> flux_dev
    loss_fn     = (m, x) -> Flux.mse(m(x), y_flux)

    # warmup
    Flux.withgradient(m -> loss_fn(m, x_flux), block_flux)
    dev isa NeuroDSL.Backend.CUDADevice && CUDA.synchronize()

    t_flux = @benchmark begin
        Flux.withgradient(m -> $loss_fn(m, $x_flux), $block_flux)
        $(dev isa NeuroDSL.Backend.CUDADevice ? :(CUDA.synchronize()) : :())
    end samples=n_samples evals=1

    med_n = median(t_neuro).time * 1e-6   # ns → ms
    med_f = median(t_flux).time  * 1e-6
    ratio = med_n / med_f

    @printf "  Flux     médiane : %8.3f ms\n"  med_f
    @printf "  NeuroDSL médiane : %8.3f ms\n"  med_n
    @printf "  Ratio (N/F)      :     %.2fx  %s\n" ratio (ratio < 1 ? "✅ NeuroDSL PLUS RAPIDE" : ratio < 1.15 ? "≈ parité" : "⚠ NeuroDSL plus lent")

    return (dim=dim, seq=seq, device=dev_name,
            flux_ms=med_f, neuro_ms=med_n, ratio=ratio)
end

# ── Série complète ─────────────────────────────────────────────────────────

println("="^55)
println("  Benchmark Transformer Block — NeuroDSL vs Flux")
println("="^55)

results = []

# GPU — les trois dimensions
# Note : on évite les noms `gpu` et `cpu` — Flux les exporte dans Main,
# toute assignation globale avec ces noms lève une erreur Julia.
gpu_dev = NeuroDSL.Backend.CUDADevice()
for dim in [256, 512, 1024]
    push!(results, bench_transformer_block(dim=dim, seq=64, dev=gpu_dev))
end

# CPU — seulement 256 et 512 (1024 CPU est trop lent pour un benchmark propre)
cpu_dev = NeuroDSL.Backend.CPUDevice()
for dim in [256, 512]
    push!(results, bench_transformer_block(dim=dim, seq=64, dev=cpu_dev, n_samples=20))
end

# ── Tableau récapitulatif ──────────────────────────────────────────────────
println("\n")
println("┌─────────────────────────────────────────────────────────┐")
println("│          Récapitulatif — médiane forward+backward        │")
println("├──────┬──────┬──────┬──────────┬──────────┬─────────────┤")
println("│  dim │  seq │  dev │  Flux ms │ NeuroDSL │    ratio    │")
println("├──────┼──────┼──────┼──────────┼──────────┼─────────────┤")
for r in results
    verdict = r.ratio < 1.0 ? "✅ +$(round((1-r.ratio)*100, digits=1))%" :
              r.ratio < 1.15 ? "≈ parité" :
              "⚠ ×$(round(r.ratio, digits=2))"
    @printf "│ %4d │  %3d │ %3s  │ %8.3f │ %8.3f │ %-11s │\n"  r.dim r.seq r.device r.flux_ms r.neuro_ms verdict
end
println("└──────┴──────┴──────┴──────────┴──────────┴─────────────┘")
println("  ✅ = NeuroDSL plus rapide  |  ⚠ = NeuroDSL plus lent")

  Benchmark Transformer Block — NeuroDSL vs Flux

━━━  dim=256  seq=64  GPU  ━━━
  Flux     médiane :   12.667 ms
  NeuroDSL médiane :    5.924 ms
  Ratio (N/F)      :     0.47x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=512  seq=64  GPU  ━━━
  Flux     médiane :    8.310 ms
  NeuroDSL médiane :    7.243 ms
  Ratio (N/F)      :     0.87x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=1024  seq=64  GPU  ━━━
  Flux     médiane :    8.426 ms
  NeuroDSL médiane :    7.866 ms
  Ratio (N/F)      :     0.93x  ✅ NeuroDSL PLUS RAPIDE

━━━  dim=256  seq=64  CPU  ━━━
  Flux     médiane :   37.786 ms
  NeuroDSL médiane :   40.134 ms
  Ratio (N/F)      :     1.06x  ≈ parité

━━━  dim=512  seq=64  CPU  ━━━
  Flux     médiane :   42.787 ms
  NeuroDSL médiane :   55.408 ms
  Ratio (N/F)      :     1.29x  ⚠ NeuroDSL plus lent


┌─────────────────────────────────────────────────────────┐
│          Récapitulatif — médiane forward+backward        │
├──────┬──────┬──────┬──────────┬──────────┬─────────────┤
│  dim │  seq │  de

# Gemma 

In [3]:
function run_neurodsl_audit()
    println("🚀 Starting NeuroDSL System Audit...\n")
    
    ns = :test_ns
    g = JuliusGraph(device=Backend.active_device())
    activate!(g, ns)
    
    x_val = randn(Float32, 4, 2)
    set!(g, :x, x_val; namespace=ns)
    W1 = randn(Float32, 3, 2)
    set!(g, :W1, W1; is_param=true, namespace=ns)
    addrule!(g, GraphRule(:z1, [:x, :W1], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    addrule!(g, GraphRule(:a1, [:z1], :relu; namespace=ns))
    W2 = randn(Float32, 1, 3)
    set!(g, :W2, W2; is_param=true, namespace=ns)
    addrule!(g, GraphRule(:z2, [:a1, :W2], :matmul; attrs=Dict(:trans_b=>true), namespace=ns))
    addrule!(g, GraphRule(:loss, [:z2], :sum_matrix; namespace=ns))

    println("✅ Graph constructed successfully in namespace :$ns.")

    println("\n--- Test 1: Operator Fusion ---")
    success = _fuse!(g, [:z1, :a1]; ns=ns)
    if success && haskey(g.rules[ns], :a1) && g.rules[ns][:a1].op == :fused_matmul_relu
        println("✅ Fusion successful: [:z1, :a1] -> :fused_matmul_relu")
    else
        println("❌ Fusion failed!")
    end

    println("\n--- Test 2: Full Backward Baseline ---")
    ctx_full = CtxStore()
    demand!(g, :loss; ctx_store=ctx_full, namespace=ns)
    backward_graph!(g, :loss; ctx_store=ctx_full, full=true, namespace=ns)
    grad_W1_full = copy(g.nodes[ns][:W1].gradient)
    println("✅ Baseline gradients captured.")

    println("\n--- Test 3: Incremental Backward Correctness ---")
    # Modify W2. This SHOULD invalidate W1 because W1 is upstream of W2.
    set!(g, :W2, W2 .* 1.1f0; is_param=true, namespace=ns) 
    
    # CORRECT EXPECTATION: W1 should now be backwarded = false
    if g.nodes[ns][:W1].backwarded == false
        println("✅ Invalidation logic correct: W1 was reset because W2 changed.")
    else
        println("❌ Invalidation logic error: W1 should have been reset.")
    end

    ctx_inc = CtxStore()
    demand!(g, :loss; ctx_store=ctx_inc, namespace=ns)
    backward_graph!(g, :loss; ctx_store=ctx_inc, full=false, namespace=ns)
    
    grad_W1_inc = g.nodes[ns][:W1].gradient

    if grad_W1_inc !== nothing && !isapprox(grad_W1_full, grad_W1_inc, atol=1e-5)
        println("✅ Incremental Gradient Match: W1 gradient updated correctly.")
    else
        println("❌ Incremental Gradient Error: W1 gradient was either wiped or didn't change.")
    end

    println("\n--- Test 4: Shape Inference Audit ---")
    try
        # CALL VIA MODULE
        shape = NeuroDSL._infer_output_shape(:fused_matmul_relu, [g.nodes[ns][:x].value, g.nodes[ns][:W1].value], Dict(:trans_b=>true))
        if shape == (4, 3)
            println("✅ Shape Inference correct: $shape")
        else
            println("❌ Shape Inference wrong: $shape")
        end
    catch e
        println("❌ Shape Inference crashed: $e")
    end

    println("\n\n🎉 Audit Complete!")
end
run_neurodsl_audit()

🚀 Starting NeuroDSL System Audit...

✅ Graph constructed successfully in namespace :test_ns.

--- Test 1: Operator Fusion ---
✅ Fusion successful: [:z1, :a1] -> :fused_matmul_relu

--- Test 2: Full Backward Baseline ---
✅ Baseline gradients captured.

--- Test 3: Incremental Backward Correctness ---
✅ Invalidation logic correct: W1 was reset because W2 changed.
✅ Incremental Gradient Match: W1 gradient updated correctly.

--- Test 4: Shape Inference Audit ---
✅ Shape Inference correct: (4, 3)


🎉 Audit Complete!


In [26]:
# 1. Créer le dossier
mkdir("figures")

# 2. Générer transformer_time.pdf
using Plots
gr()

dim = [256, 512, 1024]
flux    = [8.177, 8.424, 8.444]
neuro   = [6.549, 6.090, 5.956]
pytorch = [5.095, 5.138, 5.771]

plot(dim, [flux neuro pytorch],
     label = ["Flux" "NeuroDSL" "PyTorch"],
     marker = :circle, lw = 2,
     xlabel = "Hidden dimension",
     ylabel = "Time (ms)",
     title = "Transformer Block – Forward+Backward Time (GPU)",
     legend = :topleft)

savefig("figures/transformer_time.pdf")
println("✅ transformer_time.pdf saved")

# 3. Générer transformer_memory.pdf
neuro_mem   = [3.47, 10.35, 34.61]
pytorch_mem = [20.32, 31.38, 74.51]

plot(dim, [neuro_mem pytorch_mem],
     label = ["NeuroDSL" "PyTorch"],
     marker = :circle, lw = 2,
     xlabel = "Hidden dimension",
     ylabel = "Peak memory (MiB)",
     title = "Transformer Block – Peak GPU Memory",
     legend = :topleft)

savefig("figures/transformer_memory.pdf")
println("✅ transformer_memory.pdf saved")

✅ transformer_time.pdf saved
✅ transformer_memory.pdf saved


In [ ]:
using Plots
gr()

dim = [256, 512, 1024]
neuro_mem   = [3.47, 10.35, 34.61]
pytorch_mem = [20.32, 31.38, 74.51]

plot(dim, [neuro_mem pytorch_mem],
     label = ["NeuroDSL" "PyTorch"],
     marker = :circle, lw = 2,
     xlabel = "Hidden dimension",
     ylabel = "Peak memory (MiB)",
     title = "Transformer Block – Peak GPU Memory",
     legend = :topleft)

savefig("figures/transformer_memory.pdf")
println("✅ transformer_memory.pdf saved")